# Runtime analysis

Training cost of each predictive family, and what that cost buys. Produces
Table 19 (ms per training step, Appendix H.2), Figure 4 (cost against accuracy,
main text) and Figures 9 and 10 (the same per data set, Appendix H.2).
Everything here reads the results tree and nothing is recomputed on a GPU.

1. [The measurements](#1-the-measurements)
2. [Joint cost tables](#2-joint-cost-tables) — one row per task, one column per
   method, in `steps/s` and in `s/step`
3. [Cost against accuracy](#3-cost-against-accuracy) — the two figures
   the paper carries, one panel per metric

**The baseline is the diagonal arm.** Every relative number in section 3 is a
ratio to `phys_diag` on the *same* task: the same backbone, the same resolution,
and the cheapest head that still predicts a covariance. That makes the panels
answer *given that you are predicting a covariance, what does structure cost* —
which is the question the paper argues about. A point predictor cannot serve as
the denominator for it: it has no log score at all, and its CRPS is a different
kind of number.

**Two families are therefore not drawn in section 3**: Det, which is not a
covariance model, and Chol (`phys_cholesky`), the unstructured ceiling that is
only tractable at small $V$. Both are still measured and
both keep their column in the cost tables of section 2 — those are absolute
per-step times and carry no denominator at all.

One artifact uses a different denominator. `make_runtime_table.py --export`
writes a `rel_to_det` column into
`runtime_analysis/*/runtime.csv`. This notebook never reads it — `arm_costs`
recomputes the ratio from `runtime.jsonl` — so the two are a genuine difference
of denominator, not one being a stale copy of the other. Anything quoting that
CSV is quoting overheads against the point predictor.

**Sources.**

| what | where | written by |
|---|---|---|
| per-step cost | `<experiment>/runtime_analysis/runtime.jsonl` | `src/train/benchmark.py`, exported by `make_runtime_table.py --export` |
| test scores | `<experiment>/<dataset>/[<mode>/][T<pred_len>/]<slug>/seed<N>/metrics.json` | `tools/sync_checkpoints.py` |

The two share a path tail by construction, which is the whole join: a cost and a
score sit under the same task name because `make_runtime_table.identity` and
`sync_checkpoints.identity` derive it the same way.

In [ ]:
import json
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


def _project_root(start=None):
    """Nearest ancestor with a `.project-root` marker -- the rule plot_config uses.

    Lets the notebook run from the repository root or from `notebooks/` without
    the imports below depending on which.
    """
    start = Path.cwd() if start is None else Path(start)
    for parent in [start, *start.parents]:
        if (parent / ".project-root").exists():
            return parent
    return start


PROJECT_ROOT = _project_root()
NB_DIR = PROJECT_ROOT / "notebooks"
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import plot_config as pc
import make_runtime_table as mrt        # arm identity, shared-tree layout
import make_results_tables as mres      # slug -> facet parsing

CFG = pc.apply_style()

#: The results tree.
from src.utils.paths import configured_path
SHARED = configured_path("results_dir")

#: The sweep's own log, used only when the results-tree copy is missing or stale.
LOCAL_RUNTIME = PROJECT_ROOT / "logs" / "benchmark" / "runtime.jsonl"

#: How a configuration's seeds are collapsed to the one number a marker sits at.
#: **`"median"`**, because the mean can be a number no seed produced. SB can
#: suffer ensemble collapse on some seeds and not others -- a collapsed seed falls
#: to ~1-2 % of the correct predictive spread and becomes a point forecast (CRPS
#: equals MAE to three decimals). It does so on 2 of 5 seeds at illness T36 and on
#: all five at depth. Retraining the bad seeds is not an option -- every arm uses
#: one fixed seed set, and dropping the seeds that misbehave for one of them is
#: cherry-picking -- so the statistic has to be the robust one. With 2 of 5
#: collapsed the median still reports the arm's typical behaviour; it would flip
#: at 3 of 5, which is the right behaviour and not a failure of the statistic.
#:
#: It governs **both** axes of a section-3 marker: `arm_costs` builds
#: `rel_to_diag` from the matching cost statistic, and `cost_accuracy` uses
#: `seed_center` for the score and its reference. `TASK_STATISTIC` then
#: collapses across data sets. `"mean"` gives the mean-based figures; nothing
#: else has to change.
SEED_STATISTIC = "median"

#: How an arm's per-data-set values are collapsed to the one cross the main-paper
#: figure (Figure 4) draws: the **median**, matching the paper's caption (median
#: over seeds and data sets) and `ablations.ipynb`. This is a different question
#: from `SEED_STATISTIC` above: that one is robustness *within* a task, where one
#: arm's seed population can be bimodal, while this one summarizes *across* data
#: sets.
#:
#: The case for `"mean"` is that the extremes are where a covariance model earns
#: its keep, and with few data sets a median discards exactly the ones that
#: separate the arms. The case against it is that a mean is sensitive to the
#: composition of a convenience sample and to degenerate cells (see the log-score
#: discussion in section 3). `task_statistic_table(GRID_FRAMES)` prints every
#: cross both ways (mean, median and shift per metric and arm); it is defined in
#: section 3.1 but not called by any cell, so run it by hand when checking a
#: cross.
TASK_STATISTIC = "median"

#: What counts as **one vote** in the cross-task step. The tasks are not
#: independent: ETTm1 alone is four of them (two modes x two horizons). With
#: **`"dataset"`** an arm's per-task values are first collapsed within each data
#: set by `GROUP_STATISTIC` -- ETTm1 to the median of its four tasks, illness and
#: weather to the median of their two horizons, ERA5, depth and EUPPBench
#: unchanged -- and `TASK_STATISTIC` then runs over data sets, so every data set
#: counts once however many (mode, horizon) variants of it were run.
#: `"dataset_mode"` keeps univariate and multivariate apart and collapses only the
#: horizons; `None` is one vote per task. It applies to both figures -- the
#: appendix draws one dot per group, so the main figure's cross is still the
#: statistic of exactly the dots behind it -- and to nothing else: the coverage
#: table, the cost tables and `cost_accuracy` stay per task.
TASK_GROUPING = "dataset"

#: How the tasks of one group are collapsed. A group has one, two or four
#: tasks, so its median is the mean of its middle two at most -- close to
#: "average over the horizons" either way, without letting one diverged task
#: set the value of its data set.
GROUP_STATISTIC = "median"

#: Where the generated .tex goes. Local: these two tables are joint across
#: experiments, so there is no experiment folder in the results tree they belong in.
TABLE_DIR = NB_DIR / "tables"

## 1. The measurements

`src/train/benchmark.py` times one training step of every arm on every task, five
seeds each, all on one card. A row is one (task, method, seed, stage); the fields
this notebook uses are `step_ms_median` (the median over the timed steps of that
run), `effective_step_ms` (staged arms only), `status`, and the task facets.

Two conventions that the numbers below depend on:

* **A staged arm's headline is `effective_step_ms`**, the epoch-weighted mean of
  its two stages. Stage 2 freezes the trunk, so it is both cheaper *and* shorter;
  quoting stage 1 overstates the method and quoting stage 2 understates it.
* **The seed is the unit of replication.** The spread within one run is
  measurement noise on one model; the spread across seeds is what a rerun would
  actually see, and only the second belongs in an error bar.

In [ ]:
#: The forecast horizons the paper reports, per time-series data set. Every task
#: is at T = 96 and 192 except illness, which is 966 rows long and has its own
#: pair. The benchmark log is append-only and may carry timings at other horizons
#: (e.g. T = 720); those have no score to join to and would only put a row into
#: the cost tables that the results tables do not have. Dropped here, at the one
#: place rows enter the notebook, and counted.
REPORTED_HORIZONS = {"national_illness": (24, 36)}
DEFAULT_HORIZONS = (96, 192)


def _reported(rows):
    """`(rows at a reported horizon, {(dataset, T): n dropped})`."""
    keep, dropped = [], defaultdict(int)
    for r in rows:
        experiment, dataset, _mode, horizon = mrt.identity(r)
        if (experiment == "time_series" and horizon is not None
                and horizon not in REPORTED_HORIZONS.get(dataset, DEFAULT_HORIZONS)):
            dropped[(dataset, horizon)] += 1
            continue
        keep.append(r)
    return keep, dict(dropped)


def load_runtime(shared=SHARED, local=LOCAL_RUNTIME, verbose=True):
    """Every benchmark row, preferring the shared tree over the local sweep log.

    `make_runtime_table.py --export` copies `logs/benchmark/runtime.jsonl` into
    `<experiment>/runtime_analysis/runtime.jsonl`, split by experiment because
    the shared tree's root is not group-writable and because an experiment
    folder is meant to be readable on its own. Reading the shared copy is what
    makes this notebook reproducible for a collaborator who never ran the sweep.

    The local file is not ignored, though: if it holds rows the shared copy does
    not, the export is stale and saying so is more useful than silently
    analysing yesterday's sweep.
    """
    files = sorted(shared.glob("*/runtime_analysis/runtime.jsonl"))
    rows = [json.loads(line) for f in files
            for line in f.read_text().splitlines() if line.strip()]
    local_rows = ([json.loads(line) for line in local.read_text().splitlines()
                   if line.strip()] if local.is_file() else [])
    if not rows:
        if not local_rows:
            raise FileNotFoundError(
                f"no runtime rows: neither {shared}/*/runtime_analysis/ nor "
                f"{local}. Run runs/benchmark.sh, then "
                "`python notebooks/make_runtime_table.py --export`.")
        warnings.warn(f"no shared export found; falling back to {local}. Run "
                      "`python notebooks/make_runtime_table.py --export`.")
        rows = local_rows
    elif len(local_rows) > len(rows):
        warnings.warn(
            f"{local} has {len(local_rows)} rows but the shared export has "
            f"{len(rows)}: the export is stale. Re-run "
            "`python notebooks/make_runtime_table.py --export`.")
    rows, dropped = _reported(rows)
    if verbose and dropped:
        print(f"dropped {sum(dropped.values())} rows at unreported horizons: "
              + ", ".join(f"{d} T={h} ({n})" for (d, h), n in sorted(dropped.items())))
    gpus = sorted({str(r.get("gpu")) for r in rows if r.get("gpu")})
    if verbose:
        source = "shared tree" if files else str(local)
        print(f"{len(rows)} rows from the {source} "
              f"({len(files)} file(s)), GPU: {', '.join(gpus)}")
    if len(gpus) > 1:
        # A cost table is only one table if it was measured on one machine.
        warnings.warn(
            "rows come from more than one GPU (" + ", ".join(gpus) + "). "
            "Absolute per-step times are not comparable across cards, and the "
            "ratio to the deterministic arm only partly is -- it divides out "
            "the card for arms of similar arithmetic intensity, which these "
            "are not.")
    return rows


RUNTIME_ROWS = load_runtime()

In [ ]:
#: Arms whose transform differs by task but whose role does not. Each task runs
#: exactly one SCORE core (`whitened_*` slugs) -- Fourier on the gridded and
#: time-series tasks, the graph Fourier transform with a Matern filter on the
#: station graph -- so they share one column; which transform it was stays in the
#: `transform` column of the cost frame.
def family(method):
    """The arm's role, with the transform-specific `whitened_*` names collapsed."""
    return "ours" if str(method).startswith("whitened_") else str(method)


#: Column order of the joint tables, and the order arms are drawn in. Det first,
#: then Diag, then the structured families by increasing generality, then the
#: unstructured ceiling (Chol) and the nonparametric arm (SB). `ours_stage1` is
#: not a family `family()` ever returns -- it is the stage-1-only costing of SCORE
#: that `stage1_only_frame` builds -- but it still needs a place in this order to
#: sort next to `ours` wherever it appears.
FAMILY_ORDER = ["deterministic", "phys_diag", "phys_lora", "ours", "ours_stage1",
                "phys_cholesky", "generative"]

#: Short column heads. The full names are in `make_runtime_table.label`; these
#: have to fit seven abreast in a table that also carries a row label.
FAMILY_LABEL = {
    "deterministic": "Det",
    "phys_diag": "Diag",
    "phys_lora": r"LorD $r{=}%d$",   # filled with the rank
    "ours": "SCORE",
    "ours_stage1": "SCORE (stage 1)",
    "phys_cholesky": "Chol",
    "generative": "SB",
}

#: Row labels, as `(LaTeX, plain text)`. Keyed by the
#: `(experiment, dataset, mode)` prefix of the identity that
#: `make_runtime_table.identity` derives, so a task is named the same here, in
#: the `runtime_analysis/` path and in the checkpoint tree. Both spellings
#: are written out rather than one being stripped out of the other: `$t_{2m}$`
#: does not survive deleting its dollars and braces.
TASK_LABEL = {
    ("time_series", "etth1", "univariate"): ("ETTh1 (univ.)", "ETTh1 (univ.)"),
    ("time_series", "etth1", "multivariate"): ("ETTh1 (multiv.)", "ETTh1 (multiv.)"),
    ("time_series", "etth2", "univariate"): ("ETTh2 (univ.)", "ETTh2 (univ.)"),
    ("time_series", "etth2", "multivariate"): ("ETTh2 (multiv.)", "ETTh2 (multiv.)"),
    ("time_series", "ettm1", "univariate"): ("ETTm1 (univ.)", "ETTm1 (univ.)"),
    ("time_series", "ettm1", "multivariate"): ("ETTm1 (multiv.)", "ETTm1 (multiv.)"),
    ("time_series", "ettm2", "univariate"): ("ETTm2 (univ.)", "ETTm2 (univ.)"),
    ("time_series", "ettm2", "multivariate"): ("ETTm2 (multiv.)", "ETTm2 (multiv.)"),
    ("time_series", "weather", "multivariate"): ("Weather", "Weather"),
    ("time_series", "electricity", "multivariate"): ("Electricity", "Electricity"),
    ("time_series", "traffic", "multivariate"): ("Traffic", "Traffic"),
    # Illness carries no mode qualifier for the same reason Electricity and
    # Traffic do not: it exists in one mode only, so "(multiv.)" would
    # distinguish it from nothing. Keyed on `national_illness` -- the CSV stem,
    # which is what `identity` yields and what the checkpoint tree is named
    # after -- not on the `illness` datamodule config name.
    ("time_series", "national_illness", "multivariate"): ("Illness", "Illness"),
    ("surface_temperature", "era5", None): ("ERA5 $t_{2\\mathrm{m}}$", "ERA5 t2m"),
    ("depth", "nyu", None): ("NYU depth (low-res.)", "NYU depth (low-res.)"),
    ("depth", "nyu_bts", None): ("NYU depth (BTS)", "NYU depth (BTS)"),
    ("station_postprocessing", "euppbench", None): ("EUPPBench", "EUPPBench"),
}


def task_label(task, tex=True):
    """Row label of one task, with the forecast horizon when it has one."""
    experiment, dataset, mode, horizon = task
    pair = TASK_LABEL.get((experiment, dataset, mode))
    if pair is None:                      # a task nobody has named yet
        fallback = dataset + (f" ({mode})" if mode else "")
        pair = (fallback, fallback)
    base = pair[0] if tex else pair[1]
    if horizon:
        base += f", $T{{=}}{horizon}$" if tex else f", T={horizon}"
    return base


#: Row and figure order of the experiments: the time series first, then ERA5, NYU
#: depth and EUPPBench. An experiment not listed sorts after them, alphabetically.
EXPERIMENT_ORDER = ("time_series", "surface_temperature", "depth",
                    "station_postprocessing")

#: Time-series data sets that come after all the others, in this order:
#: electricity after weather, in the tables and in every figure, and traffic last.
TS_LAST = ("electricity", "traffic")


def task_order(task):
    """Sort key for a task: `EXPERIMENT_ORDER`, then the identity, with the horizon
    compared as a number.

    Sorting the identity as a string puts `T=720` before `T=96`, which reads as
    an accident in a printed table. Within an experiment the data sets are
    alphabetical, except that `TS_LAST` comes last.
    """
    experiment, dataset, mode, horizon = task
    rank = (EXPERIMENT_ORDER.index(experiment) if experiment in EXPERIMENT_ORDER
            else len(EXPERIMENT_ORDER))
    last = TS_LAST.index(dataset) if dataset in TS_LAST else -1
    return (rank, experiment, last, dataset, mode or "", int(horizon or 0))


def task_group(task, grouping):
    """The identity of the group `task` votes in under `grouping`.

    Still a 4-tuple shaped like a task, with the collapsed levels set to `None`,
    so everything that reads a `task` column -- the labels, the sort order, the
    `nunique` counts -- works on a grouped frame unchanged.
    """
    experiment, dataset, mode, _horizon = task
    if grouping is None:
        return task
    if grouping == "dataset":
        return (experiment, dataset, None, None)
    if grouping == "dataset_mode":
        return (experiment, dataset, mode, None)
    raise ValueError(f"unknown TASK_GROUPING {grouping!r}")


def group_label(group, tex=True):
    """Label of a group: a task label, or a data set's name without its mode."""
    experiment, dataset, mode, horizon = group
    if mode is None and (experiment, dataset, None) not in TASK_LABEL:
        # A whole data set that exists in several modes: any of its labels
        # names it once the mode qualifier is dropped.
        for (e, d, _m), pair in TASK_LABEL.items():
            if (e, d) == (experiment, dataset):
                base = pair[0] if tex else pair[1]
                return base.replace(" (univ.)", "").replace(" (multiv.)", "")
    return task_label(group, tex=tex)


def arm_label(fam, rank, tex=True):
    """Column head of one arm."""
    text = FAMILY_LABEL.get(fam, str(fam))
    if "%d" in text:
        text = text % int(rank or 0)
    return text if tex else text.replace("$", "").replace("{=}", "=")


def arm_order(arm):
    """Sort key for `(family, rank)`, following `FAMILY_ORDER` then the rank."""
    fam, rank = arm
    idx = FAMILY_ORDER.index(fam) if fam in FAMILY_ORDER else len(FAMILY_ORDER)
    return (idx, int(rank or 0))

In [ ]:
def seed_costs(rows=None, stage1=False):
    """One row per (task, arm, seed): that seed's headline per-step cost, in ms.

    The headline of a *staged* arm is `effective_step_ms`, the epoch-weighted
    mean of its two stages, which `benchmark.py` writes onto both stage rows of
    a seed -- so keying on the seed both picks the right number and collapses
    the two rows into one measurement. A single-stage arm's headline is its
    `step_ms_median`.

    `oom` rows are kept with `ms = NaN` and their status: "this configuration did
    not fit on the card" is a result the table should print, not a gap it should
    leave blank. They are the only rows with no timing at all -- `over_budget`
    rows are real measurements that merely exceeded the card the campaign
    targets.

    `stage1=True` returns the `ours` arm costed at stage 1 alone instead, as the
    family `ours_stage1`: the stage-1 row's own `step_ms_median`, what a run
    that stopped after stage 1 would pay. Section 3 reads the same number
    through `stage1_seed_ms`.
    """
    rows = RUNTIME_ROWS if rows is None else rows
    seen = {}
    unknown = defaultdict(int)
    for r in rows:
        if stage1 and not (r.get("staged") and r.get("stage") == 1
                           and family(r["method"]) == "ours"):
            continue
        try:
            task = mrt.identity(r)
        except ValueError:               # a datamodule the mapping has not met
            unknown[str(r.get("datamodule_cls"))] += 1
            continue
        key = (task, "ours_stage1" if stage1 else family(r["method"]),
               r.get("rank") or 0, r.get("seed"))
        measured = r.get("status") in mrt.MEASURED_STATUS
        ms = (r.get("step_ms_median") if stage1 or not r.get("staged")
              else r.get("effective_step_ms")) if measured else None
        seen[key] = dict(
            task=key[0], family=key[1], rank=key[2], seed=key[3],
            status=r.get("status"), ms=float(ms) if ms else np.nan,
            method=r["method"], transform=r["method"].removeprefix("whitened_")
            if str(r["method"]).startswith("whitened_") else None,
            staged=bool(r.get("staged")), batch_size=r.get("batch_size"),
            event_dim=r.get("event_dim"), peak_gb=r.get("peak_alloc_gb"),
            loss=r.get("loss"), n_train_samples=r.get("n_train_samples"))
    for dm, n in sorted(unknown.items()):
        warnings.warn(f"{n} rows with unknown datamodule {dm!r} skipped; extend "
                      "make_runtime_table._EXPERIMENT to match sync_checkpoints.")
    return pd.DataFrame(list(seen.values()))


def arm_costs(seeds=None):
    """Seed mean, median and sample sd of both rates, one row per (task, arm).

    Both rates are aggregated *from the per-seed values*, never derived from each
    other: `steps/s` is the mean of `1000/ms_i`, not `1000/mean(ms_i)`. Jensen
    separates the two, and only the first carries an error bar that means "what a
    rerun would see". The sd is the sample sd (n-1) across seeds; an arm with one
    seed gets NaN rather than a zero that would read as "no variation".

    **Mean and median are both kept, for different consumers.** Section 2's cost
    tables report `mean +- sd`, which is what their captions say; section 3's
    cost-accuracy panels use the median, so a marker there is a median on both of
    its axes (`SEED_STATISTIC` explains why the score axis has to be robust).
    Timing has no pathology to be robust against -- the seed CV of `step_ms` is
    0.5 % at the median arm and 5.2 % at the worst, and mean and median differ by
    at most 1.6 % -- so this is consistency, not correction. Not quite cosmetic
    either: the largest disagreements are on the *diagonal* arm, which is the
    denominator of every ratio in section 3, so mixing the two statistics would
    put a 1.6 % wobble under every point of that task.
    """
    seeds = seed_costs() if seeds is None else seeds
    out = []
    for (task, fam, rank), g in seeds.groupby(["task", "family", "rank"],
                                              sort=False):
        ok = g[g["ms"].notna()]
        first = g.iloc[0]
        row = dict(task=task, family=fam, rank=rank,
                   n_seeds=len(ok), n_rows=len(g),
                   status="ok" if len(ok) else str(first["status"]),
                   method=first["method"], transform=first["transform"],
                   staged=bool(first["staged"]),
                   batch_size=first["batch_size"], event_dim=first["event_dim"],
                   peak_gb=g["peak_gb"].max())
        if len(ok):
            ms = ok["ms"].to_numpy(dtype=float)
            sps = 1e3 / ms
            row.update(s_per_step=float(np.mean(ms)) / 1e3,
                       s_per_step_med=float(np.median(ms)) / 1e3,
                       s_per_step_sd=float(np.std(ms, ddof=1)) / 1e3
                       if len(ms) > 1 else np.nan,
                       # What the joint table prints: the same mean and sd in ms.
                       ms_per_step=float(np.mean(ms)),
                       ms_per_step_sd=float(np.std(ms, ddof=1))
                       if len(ms) > 1 else np.nan,
                       steps_per_s=float(np.mean(sps)),
                       steps_per_s_med=float(np.median(sps)),
                       steps_per_s_sd=float(np.std(sps, ddof=1))
                       if len(sps) > 1 else np.nan)
        else:
            row.update(s_per_step=np.nan, s_per_step_med=np.nan,
                       s_per_step_sd=np.nan, ms_per_step=np.nan,
                       ms_per_step_sd=np.nan, steps_per_s=np.nan,
                       steps_per_s_med=np.nan, steps_per_s_sd=np.nan)
        out.append(row)
    costs = pd.DataFrame(out)
    # Overhead against **Diag** on the same task -- the denominator every relative
    # statement in section 3 is made in.
    #
    # Not Det. A point predictor is not a covariance model: it has no log score at
    # all, and its CRPS is a different kind of number. The question these panels ask
    # is "given that you are predicting a covariance, what does *structure* cost",
    # and the diagonal is the cheapest arm that answers it.
    #
    # The column is named for its denominator rather than called `rel_to_ref`, so
    # that changing the denominator cannot pass unnoticed through an edit that
    # renames nothing. (`make_runtime_table.py --export` writes `rel_to_det` into
    # `runtime_analysis/*/runtime.csv`; this notebook never reads it.)
    # `cost_col` follows `SEED_STATISTIC`, so a section-3 marker is the same kind
    # of number on both axes. The absolute `s_per_step` / `steps_per_s` columns
    # are untouched, so section 2's tables keep the mean their captions promise.
    cost_col = "s_per_step_med" if SEED_STATISTIC == "median" else "s_per_step"
    ref = {t: g.loc[g["family"] == "phys_diag", cost_col].mean()
           for t, g in costs.groupby("task", sort=False)}
    costs["diag_s_per_step"] = costs["task"].map(ref)
    costs["rel_to_diag"] = costs[cost_col] / costs["diag_s_per_step"]
    costs["overhead_pct"] = 100 * (costs["rel_to_diag"] - 1)
    costs["task_label"] = costs["task"].map(lambda t: task_label(t, tex=False))
    costs["arm_label"] = [arm_label(f, r, tex=False)
                          for f, r in zip(costs["family"], costs["rank"])]
    order = {t: i for i, t in enumerate(sorted(costs["task"].unique(),
                                               key=task_order))}
    costs["_task"] = costs["task"].map(order)
    costs["_arm"] = [arm_order((f, r))
                     for f, r in zip(costs["family"], costs["rank"])]
    return (costs.sort_values(["_task", "_arm"])
                 .drop(columns=["_task", "_arm"]).reset_index(drop=True))


SEEDS = seed_costs()
COSTS = arm_costs(SEEDS)
#: SCORE costed at stage 1 alone, in `COSTS`' schema, for the last column of the
#: ms/step table (Table 19). Kept out of `COSTS`, so section 3 never sees it as an
#: arm; its `rel_to_diag` is NaN (its frame has no diagonal rows) and nothing reads it.
STAGE1_COSTS = arm_costs(seed_costs(stage1=True))
print(f"{COSTS['task'].nunique()} tasks x {len(COSTS)} arms, "
      f"{int(COSTS['n_seeds'].sum())} timed runs, "
      f"{int((COSTS['status'] != 'ok').sum())} arm(s) out of memory")
COSTS[["task_label", "arm_label", "n_seeds", "status", "batch_size", "event_dim",
       "ms_per_step", "ms_per_step_sd", "steps_per_s", "steps_per_s_sd",
       "overhead_pct"]]

## 2. Joint cost tables

One row per task, one column per method — the same numbers twice, in the two
units that answer different questions.

**`s/step`** is the honest per-iteration cost, and is what the arms *within* one
row are ranked by: they share a batch size, so a step is the same unit of work
for all of them. **`steps/s`** is its reciprocal, which is the easier number to
read when the comparison is "how much slower". Neither transfers *between* rows:
tasks differ in batch size and in event dimension, so a row is a comparison and a
column is not. Making rows comparable takes a ratio, which divides out the
backbone, the resolution and the card — that is what section 3 does, against the
diagonal arm. **These two tables are absolute times and carry no denominator**,
which is also why they keep every measured arm, including the two the panels
leave out.

Both are aggregated across seeds from the per-seed values, so `steps/s` is the
mean of the per-seed rates rather than the reciprocal of the mean cost; the
error is the sample standard deviation across seeds.

In [ ]:
#: Decimal places of every cell, mean and deviation alike: one precision across
#: all the paper's tables.
DIGITS = 3


def pm(mean, sd, digits=DIGITS, stacked=False, mean_digits=None, bold=False,
       italic=False):
    """`mean ± sd` as LaTeX math, both to `digits` decimal places.

    `mean_digits` sets the mean's precision on its own: the ms/step table prints
    whole milliseconds and keeps the deviation at `digits`. `bold` and `italic`
    set the cell (the mean alone, when stacked) in bold and in italic; both at
    once go through `\\boldmath`, because math alphabets do not nest --
    `\\mathbf{\\mathit{..}}` is plain italic.

    `stacked` puts the deviation on a second line inside the cell. It is not a
    style choice: at the significant-figure precision this table used before
    2026-09-29, seven method columns of `$0.01224 \\pm 0.00009$` were 8.8 in
    wide, half again the 6.75 in a two-column page has, and the cell is twice as
    wide as it is tall for no reason -- a stacked cell is as wide as its widest
    line and the table fits with room to spare. The alternatives both cost more
    than a line break: shrinking the font buys 5 % (the column widths are set by
    digits, and `\\tabcolsep` does not scale), and `\\resizebox` rescales the
    type away from the paper's. The ms/step table no longer needs it: without
    its $V$ and $B$ columns and with whole-millisecond means, one line fits.
    """
    if mean is None or not np.isfinite(mean):
        return None
    md = digits if mean_digits is None else mean_digits

    def b(body):
        if bold and italic:
            return rf"{{\boldmath$\mathit{{{body}}}$}}"
        if bold or italic:
            return rf"$\math{'bf' if bold else 'it'}{{{body}}}$"
        return rf"${body}$"

    if sd is None or not np.isfinite(sd):
        return b(f"{mean:.{md}f}")
    if stacked:
        return (r"\begin{tabular}[c]{@{}c@{}}" + b(f"{mean:.{md}f}") + r"\\[-1pt]"
                + rf"{{\scriptsize $\pm\,{sd:.{digits}f}$}}" + r"\end{tabular}")
    return b(rf"{mean:.{md}f} \pm {sd:.{digits}f}")


#: What a cell says when there is no number. An OOM is a measurement -- the arm
#: was run and did not fit -- so it is printed, not blanked; `--` means the arm
#: was never submitted for that task.
_OOM = r"\textsc{oom}"
_ABSENT = "--"


def table_arms(costs, quantity):
    """Column order of a joint table: `arm_order`, with SCORE and then its
    stage-1 costing moved to the right where `QUANTITY` asks for it.
    `FAMILY_ORDER` itself is left alone, since it also orders the legends of
    section 3."""
    arms = sorted({(f, r) for f, r in zip(costs["family"], costs["rank"])},
                  key=arm_order)
    if QUANTITY[quantity]["score_last"]:
        last = ("ours", "ours_stage1")
        arms = ([a for a in arms if a[0] not in last]
                + sorted((a for a in arms if a[0] in last),
                         key=lambda a: last.index(a[0])))
    return arms


def cost_matrix(costs=None, quantity="ms_per_step", tex=True, stacked=False,
                bold=False, italic=False):
    """Tasks x arms, formatted. `quantity` is `ms_per_step` or `steps_per_s`.

    LaTeX only: `bold` marks the best arm of each task among those outside
    `QUANTITY[...]["bold_skip"]`, `italic` the best arm overall. Both compare at
    the printed precision, so two arms that print the same number are both marked.
    The families in `QUANTITY[...]["unmarked"]` are printed but neither compete
    nor get marked.
    """
    costs = COSTS if costs is None else costs
    q = QUANTITY[quantity]
    tasks = sorted(costs["task"].unique(), key=task_order)
    arms = table_arms(costs, quantity)
    # A plain dict, not `set_index(...).loc[...]`: a task *is* a tuple here, and
    # pandas reads a tuple key as a multi-axis indexer rather than as one label.
    idx = {(r["task"], r["family"], r["rank"]): r for _, r in costs.iterrows()}

    def shown(row):
        return round(row[quantity], q["mean_digits"])

    best, best_kept = {}, {}
    for (task, fam, _), row in idx.items():
        if (row["status"] == "ok" and np.isfinite(row[quantity])
                and fam not in q["unmarked"]):
            best[task] = q["best"](best.get(task, shown(row)), shown(row))
            if fam not in q["bold_skip"]:
                best_kept[task] = q["best"](best_kept.get(task, shown(row)),
                                            shown(row))
    table = {}
    for fam, rank in arms:
        col = []
        for task in tasks:
            row = idx.get((task, fam, rank))
            if row is None:
                col.append(_ABSENT if tex else "--")
                continue
            if row["status"] != "ok":
                col.append(_OOM if tex else "OOM")
                continue
            marked = tex and fam not in q["unmarked"]
            cell = pm(row[quantity], row[f"{quantity}_sd"], stacked=tex and stacked,
                      mean_digits=q["mean_digits"],
                      bold=marked and bold and fam not in q["bold_skip"]
                      and shown(row) == best_kept[task],
                      italic=marked and italic and shown(row) == best[task])
            col.append(cell if tex else cell.replace("$", "").replace(r"\pm", "+/-"))
        table[arm_label(fam, rank, tex)] = col
    frame = pd.DataFrame(table, index=[task_label(t, tex) for t in tasks])
    frame.index.name = "Task"
    return frame


#: Header block, caption text and layout of each quantity, so the two tables
#: differ in exactly one place. The ms/step table (Table 19) prints the mean in
#: whole milliseconds with the deviation on the same line, has no $V$ / $B$
#: columns and puts SCORE last, followed by its stage-1 costing (`stage1`, from
#: `STAGE1_COSTS`); in bold is each task's fastest arm that models the
#: dependence (`bold_skip` leaves out the two that do not), in italic its
#: fastest arm overall -- Det on most tasks, which would otherwise take every
#: bold. steps/s keeps the stacked layout. `best` picks the fastest of two
#: printed values. `drop_datasets` leaves data sets out of a table altogether
#: (none at the moment); the frames of the notebook keep them.
#: `unmarked` columns are printed but never compared or marked: SCORE (stage 1)
#: is a costing of SCORE, not an arm, and fits no dependence -- stage 1 is the
#: CRPS fit of trunk and marginals, the core is fitted in stage 2.
QUANTITY = {
    "ms_per_step": dict(
        head=r"ms\,/\,step", lower_is="cheaper",
        blurb=r"milliseconds per training step (lower is cheaper)",
        mean_digits=0, stacked=False, dims=False, score_last=True, bold=True,
        italic=True, bold_skip=("deterministic", "phys_diag"), best=min,
        drop_datasets=(), stage1=True, unmarked=("ours_stage1",),
        tabcolsep="1.5pt"),
    "steps_per_s": dict(
        head=r"steps\,/\,s", lower_is="slower",
        blurb=r"training steps per second (higher is faster)",
        mean_digits=DIGITS, stacked=True, dims=True, score_last=False,
        bold=False, italic=False, bold_skip=(), best=max, drop_datasets=(),
        stage1=False, unmarked=(), tabcolsep="2.5pt"),
}

In [ ]:
#: The card every number came from, for the captions. `load_runtime` has
#: already warned if the rows did not all come from one.
GPU_NAME = sorted({str(r.get("gpu")) for r in RUNTIME_ROWS if r.get("gpu")})[0]


def to_latex(costs=None, quantity="ms_per_step", label=None):
    """The joint table as LaTeX: one row per task, one column per method.

    A `table*` rather than a `table`: seven method columns plus the task name do
    not fit a single column of a two-column layout, and `\\resizebox` is avoided
    on purpose -- it rescales the font away from the paper's, so a resized table
    no longer matches the text around it.

    `$V$` and `$B$` lead the numbers of the steps/s table because a per-step
    time is meaningless without them: a step is one batch of `B` cases of an
    event dimension `V`, and both differ between rows. They are what makes a row
    internally comparable and the columns of two different rows not. The ms/step
    table leaves them out (`QUANTITY[...]["dims"]`) and its caption says in
    words that they are shared along a row.
    """
    costs = COSTS if costs is None else costs
    q = QUANTITY[quantity]
    if q["stage1"]:
        costs = pd.concat([costs, STAGE1_COSTS], ignore_index=True)
    # A task is (experiment, dataset, mode, horizon).
    costs = costs[[t[1] not in q["drop_datasets"] for t in costs["task"]]]
    tasks = sorted(costs["task"].unique(), key=task_order)
    arms = table_arms(costs, quantity)
    frame = cost_matrix(costs, quantity, tex=True, stacked=q["stacked"],
                        bold=q["bold"], italic=q["italic"])
    meta = {r["task"]: r for _, r in costs.drop_duplicates("task").iterrows()}

    label = label or f"tab:runtime-{quantity.replace('_', '-')}"
    seeds = int(costs["n_seeds"].max())
    if q["dims"]:
        unit = (r"$V$ is the event dimension and $B$ the batch size, both shared "
                r"by every arm of a row: a step is the same unit of work across a "
                r"row and not across a column, so the comparison is within a row.")
    else:
        unit = (r"Every arm of a row shares its event dimension and batch size, so "
                r"a step is the same unit of work across a row and not across a "
                r"column, and the comparison is within a row.")
    # The OOM legend only where the table prints one.
    marks = r"\emph{--} marks an arm that does not exist for that task."
    if (costs["status"] != "ok").any():
        marks = (r"\textsc{oom} marks an arm that was measured and did not fit; "
                 r"\emph{--} an arm that does not exist for that task.")
    if q["bold"]:
        skipped = " and ".join(arm_label(f, 0) for f in q["bold_skip"])
        unit += (rf" Bold is the fastest arm of each task other than {skipped}, "
                 r"i.e.\ among the arms that model the dependence between "
                 r"coordinates" + ("; italic is the fastest arm overall."
                                   if q["italic"] else "."))
    stage1_note = (r" SCORE (stage 1) is the per-step cost of its first stage "
                   r"alone, the CRPS fit before the core is fitted; it is part of "
                   r"SCORE rather than an arm of its own, so it is not marked."
                   if q["stage1"] else "")

    def head(label):
        # "SCORE (stage 1)" on two lines, bottom-aligned with the one-line heads:
        # on one line it is wider than any number in its column.
        name, sep, rest = label.partition(" (")
        if not sep:
            return label
        return r"\begin{tabular}[b]{@{}c@{}}" + name + r"\\(" + rest + r"\end{tabular}"

    out = [
        r"\begin{table*}[t]",
        r"  \centering",
        rf"  \caption{{Training cost of every predictive family on every "
        rf"benchmark task, in {q['blurb']}. Mean $\pm$ sample standard "
        rf"deviation over {seeds} seeds, all measured on one {GPU_NAME} with "
        rf"\texttt{{src/train/benchmark.py}}. {unit} SCORE is staged, so its entry is the "
        rf"epoch-weighted mean of the two stage costs.{stage1_note} {marks}}}",
        rf"  \label{{{label}}}",
        r"  \footnotesize",
        # Per quantity, not the 6pt default: at \footnotesize the ms/step table
        # (eight arms, one line each) is 6.68 in at 1.5pt -- 6.93 in at 2.5pt --
        # and the stacked steps/s table 5.54 in at 2.5pt, against a 6.75 in text
        # width (pdflatex, Computer Modern; Times is narrower, 6.50 and 5.22 in).
        rf"  \setlength{{\tabcolsep}}{{{q['tabcolsep']}}}",
        r"  \begin{tabular}{l " + ("r r " if q["dims"] else "") + "c" * len(arms) + "}",
        r"    \toprule",
        "    Task & " + ("$V$ & $B$ & " if q["dims"] else "")
        + " & ".join(head(arm_label(f, r)) for f, r in arms) + r" \\",
        r"    \midrule",
    ]
    for task, row_label in zip(tasks, frame.index):
        m = meta[task]
        cells = [frame.loc[row_label, arm_label(f, r)] for f, r in arms]
        # `\,` rather than a comma as the thousands separator: 35{,}200 in math
        # mode sets the comma as a binary operator, with the wrong spacing.
        dim = f"{int(m['event_dim']):,}".replace(",", r"\,")
        lead = f"${dim}$ & {int(m['batch_size'])} & " if q["dims"] else ""
        out.append(f"    {row_label} & {lead}" + " & ".join(cells) + r" \\")
    out += [r"    \bottomrule", r"  \end{tabular}", r"\end{table*}"]
    return "\n".join(out)

In [ ]:
# The two joint tables. Printed as LaTeX and shown as a frame, and saved next to
# the paper's other generated tables.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
for quantity in ("steps_per_s", "ms_per_step"):
    tex = to_latex(COSTS, quantity)
    path = TABLE_DIR / f"runtime_{quantity}.tex"
    path.write_text(tex + "\n")
    print(f"=== {QUANTITY[quantity]['blurb']}  ->  {path.relative_to(PROJECT_ROOT)}\n")
    print(tex)
    print()

In [ ]:
# The same two tables as frames, which is the readable form in the notebook.
display(cost_matrix(COSTS, "steps_per_s", tex=False))
display(cost_matrix(COSTS, "ms_per_step", tex=False))

## 3. Cost against accuracy

One panel per metric. The horizontal axis is what an arm costs, as its per-step
overhead over the **diagonal** arm of the same task; the vertical axis is what it
scores, relative to the same reference. Both axes share one denominator, so they
divide out the backbone, the resolution and the card, and a task with a 0.27 s
step sits on the same axes as one with a 0.011 s step. A point to the left of the
guide is an arm that is *cheaper* than a diagonal head, which is not a
contradiction: SCORE uses a two-stage fit, and its stage 2 runs at less than half the
per-step cost of stage 1.

**Two figures come out of this section, and they differ only in what an arm
draws.** The main-paper grid puts **one cross per arm**, at its median
(`TASK_STATISTIC`) over the data sets it has both a timing and a score for, each
data set's tasks collapsed first (`TASK_GROUPING`); the appendix figure puts
**one dot per data set** and no cross. The split is a decision about density, not
about content: five arms over eleven data sets are fifty-five markers on
a panel a third of a column wide, and the clouds overlap into a smear exactly where the arms are
being compared. The main figure says where each arm sits; the appendix figure
says what it sits on top of. Everything else — the cells, the normalizations,
the reference — is one code path, so the second cannot drift away from the
first.

Colour alone separates the arms in both: shape would encode the same fact twice
and a diamond among circles reads as a different kind of thing rather than as a
different method. There is no box and there are no whiskers — with eleven data
sets per arm, a quartile would be an interpolation between two or three
of them. At this
sample size the markers *are* the distribution, which is the reason the appendix figure exists at all.

The diagonal arm is the reference on both axes, so every one of its markers
lands where the two guides cross — at $1\times$ and, on the percentage scale
both figures use, at $0\,\%$. That is construction, not measurement; the
interesting thing about that arm is only that the others are placed against it.

**What limits these figures is the join, not the runtime sweep.** A point exists
only where a task has *both* a timed arm and a synced `metrics.json` for the same
`(method, rank)`, at the objective that arm was timed with. The coverage table
below is the honest statement of which cells that leaves.

In [ ]:
#: Test scores carried by every `metrics.json`. Non-finite entries (a Dirac's
#: log score) are stored as null there and read back as NaN.
_RAW_METRICS = ("nll", "crps", "energy_score", "kernel_score",
                "variogram_score", "mse", "mae")

#: Metrics a panel can be drawn for. `scale` says how a task's raw value is made
#: comparable to another task's:
#:
#: * `ratio` -- divide by the reference arm. Right for the positive proper
#:   scores, where the reference is a magnitude and the quotient is unit-free.
#: * `difference` -- subtract it. The log score is not positive: it is about
#:   -1.3 per coordinate on ERA5 and +1.5 on EUPPBench, so a quotient would flip
#:   sign between tasks and the two would not belong on one axis. A difference in
#:   nats per coordinate does.
#:
#: Lower is better on every one of them, in both scales.
#: The reference is `phys_diag` (Diag) on every metric, matching the cost axis
#: (see `arm_costs`), so both axes of a panel share one denominator and a point
#: reads as one statement about one arm.
METRICS = {
    "crps": dict(label="CRPS", scale="ratio", reference="phys_diag"),
    "kernel_score": dict(label="kernel score", scale="ratio",
                         reference="phys_diag"),
    "energy_score": dict(label="energy score", scale="ratio",
                         reference="phys_diag"),
    "variogram_score": dict(label="variogram score", scale="ratio",
                            reference="phys_diag"),
    "mse": dict(label="MSE", scale="ratio", reference="phys_diag"),
    "nll_per_d": dict(label=r"log score / $d$", scale="difference",
                      reference="phys_diag"),
}

#: The objective each family is *timed* with by `runs/benchmark.sh`, and
#: therefore the only objective whose score belongs on the same point as that
#: timing. Everything not listed is timed with the kernel score.
PREFERRED_OBJECTIVE = {"deterministic": "mse"}
DEFAULT_OBJECTIVE = "kernel_score"

#: Stage-1 losses of a two-stage arm, most preferred first. The default marginal
#: kernel score is the configuration the paper reports; `crps` is the variant.
STAGE1_PREFERENCE = (None, "marginal_kernel_score", "crps")


def preferred_objective(fam):
    return PREFERRED_OBJECTIVE.get(fam, DEFAULT_OBJECTIVE)


def tree_path(task, shared=SHARED):
    """Checkpoint folder of a task: the same tail its `runtime_analysis/` uses."""
    experiment, dataset, mode, horizon = task
    path = shared / experiment / dataset
    if mode:
        path = path / mode
    if horizon:
        path = path / f"T{horizon}"
    return path


def read_scores(task, shared=SHARED):
    """One row per synced seed of a task, from its `metrics.json` files.

    The facets come from the config-slug directory name via the same parser the
    paper's tables use, so the slug convention is defined in exactly one place.
    A seed folder without `metrics.json` is invisible here -- checkpoints synced
    before 2026-08-14 have none, and re-running `tools/sync_checkpoints.py` for
    those run dirs adds the file without retraining anything.
    """
    rows, pending = [], 0
    path = tree_path(task, shared)
    if not path.is_dir():
        return pd.DataFrame(), 0
    for slug_dir in sorted(path.iterdir()):
        if not slug_dir.is_dir():
            continue
        cfg = mres.Config(mres._slug_to_tags(slug_dir.name))
        for seed_dir in sorted(slug_dir.glob("seed*")):
            mfile = seed_dir / "metrics.json"
            if not mfile.is_file():
                pending += 1
                continue
            test = json.loads(mfile.read_text()).get("test", {})
            row = dict(task=task, slug=slug_dir.name, family=family(cfg.method),
                       method=cfg.method, rank=int(cfg.rank) if cfg.rank else 0,
                       stage1=cfg.stage1, objective=cfg.objective(),
                       seed=seed_dir.name[len("seed"):])
            row.update({m: mres._num(test.get(m)) for m in _RAW_METRICS})
            rows.append(row)
    return pd.DataFrame(rows), pending

In [ ]:
def scores_for(tasks=None, verbose=True):
    """Every synced run of the tasks the runtime sweep covers, plus a coverage note.

    Not cached: this is a walk over a few dozen small JSON files, cheaper to
    re-read than to reason about whether a cache predates the last sync.
    """
    tasks = sorted(COSTS["task"].unique(), key=task_order) if tasks is None else tasks
    frames, notes = [], []
    for task in tasks:
        frame, pending = read_scores(task)
        path = tree_path(task)
        if not path.is_dir():
            notes.append((task, "no checkpoint folder", 0, 0))
        elif frame.empty:
            notes.append((task, f"{pending} seed dirs, none with metrics.json",
                          0, pending))
        else:
            notes.append((task, "ok", len(frame), pending))
            # `d` for the per-coordinate log score. Taken from the *runtime*
            # rows, which record `datamodule.get_total_dimension()` for the same
            # task -- the checkpoint's own metrics.json does not carry it.
            dim = COSTS.loc[COSTS["task"] == task, "event_dim"].iloc[0]
            frame = frame.copy()
            frame["d"] = dim
            frame["nll_per_d"] = frame["nll"] / dim
            frames.append(frame)
    if verbose:
        for task, state, n, pending in notes:
            print(f"[{task_label(task, tex=False)}] {state}"
                  + (f": {n} runs" if n else "")
                  + (f", {pending} without metrics.json" if pending else ""))
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), notes


SCORES, SCORE_NOTES = scores_for()

In [ ]:
#: Families the cost-accuracy panels leave out. Both are still timed, still in
#: `COSTS`, and still carry their column in the joint cost tables of section 2 --
#: they are dropped from the *panel* only.
#:
#: * `deterministic` (Det) is not a covariance model. It has no log score, and on
#:   the other panels it would sit alone at the top of the score axis and set the
#:   range, compressing the arms actually being compared into a band.
#: * `phys_cholesky` (Chol) is the unstructured ceiling. It is tractable only on
#:   small-V tasks (EUPPBench and univariate ETTh), so a cross over them would
#:   read as a claim about the ceiling in general that no gridded comparison can
#:   support.
#:
#: Removing a family here changes what the panels and `coverage` show and
#: nothing else: both read the frame `cost_accuracy` returns.
PANEL_EXCLUDE = {"deterministic", "phys_cholesky"}


def match_arm(scores, task, fam, rank, strict=True):
    """The synced run that belongs on the same point as a timed arm, or None.

    Matching is on `(family, rank)` and then on the *objective*. The objective is
    not a detail: `runs/benchmark.sh` times every covariance arm under the kernel
    score, and on ERA5 the NLL-fitted low-rank core scores a CRPS of 0.96 against
    the KS-fitted one's 0.15. Putting the first next to a kernel-score timing
    would draw a six-fold accuracy gap that is a fact about the objective and not
    about the method or its cost.

    With `strict=False` the closest available objective is used instead and the
    result is flagged `matched=False`, so a mismatched cell can be shown as long
    as it is shown as mismatched.

    Returns a dict with the chosen slug, its objective and its per-seed values.
    """
    want = preferred_objective(fam)
    sel = scores[(scores["task"] == task) & (scores["family"] == fam)
                 & (scores["rank"] == rank)]
    if sel.empty:
        return None
    groups = [(slug, g) for slug, g in sel.groupby("slug", sort=False)]

    def key(item):
        _, g = item
        obj, stage1 = g["objective"].iloc[0], g["stage1"].iloc[0]
        stage1 = stage1 if stage1 in STAGE1_PREFERENCE else "zzz"
        return (obj != want, STAGE1_PREFERENCE.index(stage1)
                if stage1 in STAGE1_PREFERENCE else len(STAGE1_PREFERENCE))

    groups.sort(key=key)
    slug, g = groups[0]
    matched = g["objective"].iloc[0] == want
    if strict and not matched:
        return None
    return dict(slug=slug, objective=g["objective"].iloc[0],
                stage1=g["stage1"].iloc[0], matched=matched, values=g)


def _center(values, how):
    """Median or mean of the finite entries of `values`, NaN when there are none.

    `np.median` of an empty selection is NaN *and* a RuntimeWarning; the NaN is
    what both callers want (the variogram score is not computed on the gridded
    tasks, and a sample-based arm has no log score) and the warning is noise, so
    the empty case is answered here instead of being raised and then discarded.
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if not v.size:
        return np.nan
    return float(np.median(v)) if how == "median" else float(v.mean())


def seed_center(values, how=None):
    """Collapse a configuration's seeds to one number, NaN when none reported.

    `how` defaults to `SEED_STATISTIC`. Applied to the *raw* metric of both the
    arm and its reference, so a marker sits at a ratio of two per-task medians
    rather than at the median of a ratio: the reference is one number per task,
    not a paired sample, so there is no per-seed ratio to take a median of.
    """
    return _center(values, how or SEED_STATISTIC)


def cost_accuracy(metric, strict=True, costs=None, scores=None, verbose=False,
                  per_seed=False):
    """One row per (task, arm) that has both a timing and a matching score.

    `x` is the per-step overhead over the deterministic arm of the task, in per
    cent. `y` is the metric made comparable across tasks by that metric's own
    `scale` -- a ratio to the reference arm for the positive scores, a difference
    for the log score. Seeds are collapsed by `seed_center` -- the **median** --
    before the ratio, for both the arm and its reference. That is deliberately
    *not* what the paper's tables quote, which is the seed mean; see
    `SEED_STATISTIC` for the arm whose collapse forces it. `n_seeds` still
    records how many seeds are behind each number.

    `per_seed=True` emits one row per finite seed of each cell instead, each
    against the same (median) reference -- the reference is one number per
    task, see `seed_center`. `x` stays the cell's cost: the benchmark seeds are
    not the training seeds, so there is no per-seed cost to pair a score with.
    """
    costs = COSTS if costs is None else costs
    scores = SCORES if scores is None else scores
    spec = METRICS[metric]
    rows, dropped = [], []
    have_scores = set(scores["task"]) if len(scores) else set()
    for task, cost_group in costs.groupby("task", sort=False):
        if task not in have_scores:
            # The whole task is missing, not one arm of it: say so once instead
            # of blaming the reference arm for the absence of every arm.
            dropped.append((task, None, None, "no synced scores for this task"))
            continue
        ref = match_arm(scores, task, spec["reference"], 0, strict=True)
        if ref is None:
            dropped.append((task, None, None,
                            f"no {arm_label(spec['reference'], 0, False)} "
                            "reference run"))
            continue
        ref_value = seed_center(ref["values"][metric])
        if not np.isfinite(ref_value):
            dropped.append((task, None, None,
                            f"{spec['reference']} has no finite {metric}"))
            continue
        if spec["scale"] == "ratio" and ref_value <= 0:
            # A quotient by a negative reference mirrors the axis: "closer to the
            # reference" would read as "further from it".
            dropped.append((task, None, None,
                            f"reference {metric} = {ref_value:.4g} is not positive"))
            continue
        for _, arm in cost_group.iterrows():
            fam, rank = arm["family"], arm["rank"]
            if fam in PANEL_EXCLUDE:
                # Not a `dropped` row: those are gaps in the campaign that the
                # coverage note asks the reader to close, and this is a choice.
                continue
            if arm["status"] != "ok":
                dropped.append((task, fam, rank, f"cost is {arm['status']}"))
                continue
            hit = match_arm(scores, task, fam, rank, strict=strict)
            if hit is None:
                loose = match_arm(scores, task, fam, rank, strict=False)
                why = (f"only fitted with {loose['objective']}, timed with "
                       f"{preferred_objective(fam)}") if loose else "no synced run"
                dropped.append((task, fam, rank, why))
                continue
            value = seed_center(hit["values"][metric])
            if not np.isfinite(value):
                dropped.append((task, fam, rank, f"no finite {metric}"))
                continue
            y = value / ref_value if spec["scale"] == "ratio" else value - ref_value
            row = dict(
                task=task, task_label=task_label(task, tex=False), family=fam,
                rank=rank, arm_label=arm_label(fam, rank, tex=False),
                slug=hit["slug"], objective=hit["objective"],
                objective_matched=hit["matched"], n_seeds=len(hit["values"]),
                raw=value, reference=ref_value, y=y,
                # Both cost views, so the panel can choose: the ratio reads on a
                # log axis (a 5x arm and a 1.01x arm are both legible), the
                # overhead is the same number in the additive form.
                x=arm["rel_to_diag"], overhead_pct=arm["overhead_pct"], seed=None)
            if not per_seed:
                rows.append(row)
                continue
            for seed, v in zip(hit["values"]["seed"], hit["values"][metric]):
                v = float(v)
                if np.isfinite(v):
                    rows.append({**row, "seed": seed, "raw": v,
                                 "y": v / ref_value if spec["scale"] == "ratio"
                                 else v - ref_value})
    frame = pd.DataFrame(rows)
    drop_frame = pd.DataFrame(dropped, columns=["task", "family", "rank", "why"])
    if verbose and not drop_frame.empty:
        for _, d in drop_frame.iterrows():
            arm = ("--" if not isinstance(d["family"], str)
                   else arm_label(d["family"], d["rank"], False))
            print(f"  dropped {task_label(d['task'], False)} / {arm}: {d['why']}")
    return frame, drop_frame


def coverage(metric="crps", strict=True):
    """Which (task, arm) cells the panel has, as a tasks x arms grid of seed counts.

    `0` is a cell the figure cannot draw. Printed rather than hidden: the
    campaign is still filling in, and a box resting on two tasks looks exactly
    like a box resting on five unless the counts are on the page.
    """
    frame, _ = cost_accuracy(metric, strict=strict)
    if frame.empty:
        return pd.DataFrame()
    grid = frame.pivot_table(index="task_label", columns="arm_label",
                             values="n_seeds", aggfunc="sum", fill_value=0)
    arms = sorted({(f, r) for f, r in zip(COSTS["family"], COSTS["rank"])},
                  key=arm_order)
    cols = [arm_label(f, r, tex=False) for f, r in arms]
    return grid.reindex(columns=[c for c in cols if c in grid.columns])


def group_tasks(frame, grouping="default"):
    """`frame` with one row per (group, arm) instead of one per (task, arm).

    `grouping` defaults to `TASK_GROUPING`. `x` and `y` are collapsed over the
    group's tasks with `GROUP_STATISTIC`, each on its own -- the same rule as
    `arm_centers`, one level down -- and `task` / `task_label` become the
    group's. `n_tasks` is how many tasks a row stands for and `n_seeds` their
    seeds in total; `raw` and `reference` are per-task quantities with no group
    value and are blanked rather than left holding one task's number. A
    per-seed frame is grouped per seed, so its (task, seed) dots become
    (data set, seed) dots. `grouping=None` returns the frame as it is, with
    `n_tasks = 1`.
    """
    grouping = TASK_GROUPING if grouping == "default" else grouping
    if grouping is None or frame.empty:
        return frame.assign(n_tasks=1)
    f = frame.assign(_group=frame["task"].map(lambda t: task_group(t, grouping)))
    keys = ["_group", "family", "rank"] + (["seed"] if f["seed"].notna().any()
                                          else [])
    rows = []
    for _, g in f.groupby(keys, sort=False, dropna=False):
        first = g.iloc[0].to_dict()
        x = _center(g["x"], GROUP_STATISTIC)
        rows.append({**first, "task": first["_group"],
                     "task_label": group_label(first["_group"], tex=False),
                     "x": x, "y": _center(g["y"], GROUP_STATISTIC),
                     "overhead_pct": 100.0 * (x - 1.0),
                     "raw": np.nan, "reference": np.nan,
                     "n_seeds": int(g["n_seeds"].sum()),
                     "n_tasks": int(g["task"].nunique()),
                     "objective_matched": bool(g["objective_matched"].all())})
    return pd.DataFrame(rows).drop(columns="_group")

In [ ]:
COVERAGE = coverage("crps", strict=True)
print("seeds behind each (task, arm) cell of the loss-matched panel; "
      "0 or absent = the panel cannot draw it\n")
display(COVERAGE)

_, DROPPED = cost_accuracy("crps", strict=True, verbose=True)

**All twenty-seven timed tasks reach the panel, all of them as full rows** — all
five drawn arms at five seeds (`COVERAGE` above). The timings are filtered to the
reported horizons first (`REPORTED_HORIZONS`: $T \in \{96, 192\}$, illness
$\{24, 36\}$), so a timing at any other horizon, which has no score to join to,
never enters. Traffic is timed at batch size 8 (`runs/benchmark_traffic.sh`):
at the datamodule's 24, SB does not fit on a 48 GB card. Depth is timed on
`depth_bts_crop` ($V = 238\,560$, the Eigen crop) at batch size 8, the same
task its scores come from — the cost of a covariance head scales with $V$, so
timing and score must describe the same resolution.

**The panels count data sets, not tasks** (`TASK_GROUPING = "dataset"`). The
twenty-seven tasks are eleven data sets — ETTh1, ETTh2, ETTm1 and ETTm2 with four
tasks each (two modes, two horizons), illness, weather, electricity and traffic
with two horizons each, ERA5, depth and EUPPBench with one — and an arm's
per-task values are collapsed within each data set by their median
(`GROUP_STATISTIC`) before its cross is taken, so ETT is four votes of eleven
rather than sixteen of twenty-seven. Numbers quoted *per task* below are per
task; every cross is over data sets. A median over two or four tasks is the mean
of the middle two, so a data set whose modes behave differently lands between
them (below); `TASK_GROUPING = "dataset_mode"` keeps univariate and multivariate
apart (fifteen votes), and `None` restores one vote per task.

**Depth is the most expensive task on the grid.** Its own ratios to Diag are
$1.36\times$ for SCORE, $1.57\times$ for LorD $r{=}1$, $3.21\times$ for LorD
$r{=}16$ and $3.16\times$ for SB: at $V = 238\,560$ a covariance head is no
longer a rounding error on the backbone. Over the eleven data sets the cost
crosses are SCORE $1.06\times$ (median; mean $1.07$), SB $1.64\times$ (mean
$1.77$), LorD $r{=}1$ $1.38\times$ (mean $1.37$) and LorD $r{=}16$ $1.54\times$
(mean $1.63$). Four of the eleven data sets are ETT at $V \le 1344$, where a
low-rank head costs $1.2$–$1.9\times$ Diag per task, weather ($V = 2016$ and
$4032$) costs $1.4$–$1.6\times$, electricity ($V = 30\,816$ and $61\,632$)
$1.4$–$1.7\times$ and traffic ($V = 82\,752$ and $165\,504$) $1.4$–$2.1\times$,
against illness's $1.7\times$.

**SCORE sits just above $1\times$, not below it.** It is *cheaper* than Diag
where freezing the backbone in stage 2 more than pays for the core —
$0.85\times$ on EUPPBench and $0.87\times$ on ERA5 — and within a few per cent of
it on the small-$V$ series, where there is almost no backbone cost to save:
$0.97$–$1.15\times$ over the eighteen ETT and illness tasks; weather and traffic
are at $1.16\times$ and electricity at $1.17\times$. Depth adds a third regime
and it is the unfavourable one: $V$ large enough that the core is expensive, on
a backbone heavy enough that stage 2's saving does not cover it. The defensible
summary is not "cheaper than the diagonal" but **"within about 6 % of the
diagonal, and cheaper than every other structured arm on every task measured"**
— on all twenty-seven tasks SCORE is left of LorD $r{=}1$, LorD $r{=}16$ and SB.

**SB on depth is a collapse, not a weak model**: its `crps/mae` is $0.9997$ on
all five seeds against Diag's $0.76$–$0.77$, so it has become the deterministic
forecast, landing within $1\,\%$ of `deterministic_mse` on CRPS, MAE and MSE
alike. Unlike illness $T = 36$, where 2 of 5 seeds collapse and the seed median
still reports the healthy mode, **depth has no healthy seed**, so
`SEED_STATISTIC = "median"` cannot rescue that row. Quote the collapse whenever
depth's SB point is quoted: its $-30.2\,\%$ on CRPS and $-32.1\,\%$ on ES are the
cost of a degenerate ensemble, not of a badly fitted covariance. **The same
collapse reaches the panel from the four univariate ETTm tasks**, where SB is a
point forecast on all twenty seeds (`crps/mae` $0.99$–$1.00$) and lands
$33$–$36\,\%$ below Diag on CRPS, while on the multivariate halves of the same
two data sets it is healthy and only $2$–$10\,\%$ behind. **This is where the
data-set median bites:** ETTm1 and ETTm2 land between the regimes ($-18.8$ and
$-18.6\,\%$ on CRPS), a number neither mode produced; `"dataset_mode"` would
show the two. ETTh2 multivariate is a near-collapse, and on ETTh1 SB sits far
below Diag without collapsing (a stable over-dispersion on every seed). SB's
data sets therefore split into five far below Diag (depth, ETTm1, ETTm2, ETTh2,
ETTh1, $-8.5$ to $-32\,\%$) and six near or above it, so the median — the sixth
value — sits on the healthy side of that gap ($+0.6\,\%$ on CRPS, $-0.8\,\%$ on
ES), while the means ($-6.3$ and $-8.7\,\%$) say what the gap does to the whole.

⚠️ **The log-score panel needs a decision, not just a caption.** SCORE is the
best arm on eighteen of the twenty-seven tasks ($+0.25$ to $+1.76$ nats/dim),
last but close on the two traffic tasks ($-0.50$ and $-0.56$: every fit there is
finite, SCORE at $+1.3$ to $+1.4$ nats/dim against $+0.84$ to $+0.87$ for the
kernel-score Diag), and far the worst on the other seven: depth ($-60.9$
nats/dim), the two multivariate ETTm2 tasks ($-195$ and $-3.3 \times 10^{6}$),
the two weather tasks ($-61.1$ and $-69.9$) and the two electricity tasks
($-8.1 \times 10^{3}$ and $-2.3 \times 10^{4}$). As data sets SCORE leads on six
of eleven; the cross is $+0.25$ as a median (EUPPBench's own value) and $-1421$
as a mean. On ETTm2 and weather every kernel-score fit is inflated (the
kernel-score Diag reference itself sits at $+43$ to $+228$ nats/dim on ETTm2 and
electricity, and every kernel-score fit on weather at $+230$ to $+920$, against
$+0.1$ to $+7$ for the NLL fits); on electricity and depth the SCORE core also
diverges on individual seeds (depth: $7.1$, $18.1$, $59.2$, $484.7$ and
$17\,974$ nats/dim, against a Diag flat at $-1.69$). Nothing is mismatched — the
panel pairs a kernel-score timing with a kernel-score fit by construction — and
nothing is wrong with the core as such: the *NLL-fitted* sibling
`whitened_spectral_diag_s1crps_nll` sits at $-4.78$ on depth, stable to $0.04$
across seeds, and beats the NLL-fitted Diag by about six nats. What the cell
measures is that **a kernel-score fit does not constrain the density**, and that
at large $V$ a few near-zero directions in the core are enough to make the
Gaussian log density explode. Three ways to handle it: drop `nll_per_d` for the
degenerate rows (the results tables and `ablations.ipynb` do, by
`make_results_tables.NLL_PER_D_LIMIT` applied all-or-nothing per configuration,
which removes depth, the two ETTm2 cells and both electricity cells but **not**
weather, whose seeds all sit under the limit), put that one panel on the median
across data sets, or keep the mean and say in the caption what the cross
contains. The main figure's cross is the median; the appendix figure has no
cross to take a median of, so only the first way removes those dots. Section 3.1
repeats the warning at the figure.

---

**Rank match.** `runs/benchmark.sh` times the SCORE core at **rank 0** (no
low-rank correction), so the timing is paired with the rank-free
`whitened_spectral_diag_s1crps_kernel_score` run; pairing a rank-0 cost with a
rank-$r$ score would be the same category of error as pairing timings and scores
at different resolutions. Every task trains that rank-free run and a
`generative_kernel_score` arm, which is exactly what `match_arm` looks for. SB is
timed under the fair kernel score, the objective it is fitted with, so it joins on
a loss-matched cell. `phys_diag` is the denominator of both axes and so the one
arm whose seed count every other point of its task inherits.

**On the two statistics.** Seeds are collapsed with the **median**
(`SEED_STATISTIC`), because SB's seed population can be bimodal (illness
$T = 36$, depth); the tasks of a data set with `GROUP_STATISTIC` and data sets
with `TASK_STATISTIC`, both the **median**. `task_statistic_table(GRID_FRAMES)`
prints every cross both ways. Read it before quoting a number: the NLL cross of
SCORE is $-1421$ as a mean and $+0.25$ as a median; LorD $r{=}1$'s is $+5.0$
against $+0.10$ and LorD $r{=}16$'s $+0.65$ against $+0.05$; the next largest
gaps are SB on ES ($7.9$ points), CRPS ($6.9$), VS ($3.5$), MSE ($1.5$) and KS
($1.2$), then SCORE on CRPS ($2.8$, traffic's $+26\,\%$ lifting the mean) and MSE
($1.2$), and every other cross moves by under $1$.

Reading a cross as "what this arm costs" is only safe while the task set behind
it is stated, which is what the coverage table above is for. Neither figure
prints those counts in its key: they would have to be the same number in all six
panels to be printed once, which is only true by accident.

Chol (`phys_cholesky`) is trained under both losses at five seeds on EUPPBench
and on every univariate ETTh task (not on ETTm). It is in the tree and in the
cost tables, and it is deliberately not on these panels (`PANEL_EXCLUDE`).

In [ ]:
#: One marker for every arm: the colour alone says which method a point is.
#: Shape would encode the same fact twice, and with seven arms the second
#: encoding costs more than it gives -- a diamond among circles reads as a
#: different *kind* of thing rather than as a different method, and the eye has
#: to learn a second key to read one axis.
POINT_MARKER = "o"

#: The `TASK_STATISTIC` over tasks, drawn as a cross in the arm's own colour.
CENTER_MARKER = "+"


def _lighten(hex_color, amount):
    """`hex_color` blended toward white by `amount` in [0, 1].

    Used for `ours_stage1`: it is the *same* arm as `ours`, only recosted at
    stage 1 alone, so it gets a tint of the same green rather than a new hue --
    a new colour would read as a different method.
    """
    r, g, b = mcolors.to_rgb(hex_color)
    return mcolors.to_hex((r + (1 - r) * amount, g + (1 - g) * amount,
                           b + (1 - b) * amount))


#: Per-arm colour. Where a semantic role from `plot_config.COLORS` applies it is
#: reused, so a colour keeps its meaning across the paper's figures: Det is the
#: deterministic blue, SCORE the kernel-score green it is fitted with, SB the
#: energy-score pink, and Chol the log-score purple.
FAMILY_COLOR = {
    "deterministic": pc.COLORS["deterministic"],
    "phys_diag": "0.45",
    "phys_lora": pc.COLORS["lora"],
    "ours": pc.COLORS["kernel_score"],
    "ours_stage1": _lighten(pc.COLORS["kernel_score"], 0.55),
    "phys_cholesky": pc.COLORS["log_score"],
    "generative": pc.COLORS["energy_score"],
}

#: Rank shades of LorD, light to dark. Colour is the only thing separating the
#: arms, so the two ranks that the benchmark measures take the two ends of the
#: LorD ramp rather than neighbouring browns.
LORA_SHADE = {1: pc.COLORS["lora_light"], 2: "#e6ab02",
              4: pc.COLORS["lora"], 8: pc.COLORS["lora_dark"],
              16: pc.COLORS["lora"]}


def arm_color(fam, rank):
    if fam == "phys_lora" and rank in LORA_SHADE:
        return LORA_SHADE[rank]
    return FAMILY_COLOR.get(fam, "0.3")


def task_center(values, how=None):
    """Collapse an arm's per-task values to the one number its cross sits at.

    `how` defaults to `TASK_STATISTIC`, whose trade-off is written out there.
    The same statistic on both axes: a cross drawn at a median cost and a mean
    score would sit at a pair no task produced and none summarises.
    """
    return _center(values, how or TASK_STATISTIC)


def arm_centers(frame):
    """Cost and score of each arm, collapsed over the tasks it has both for.

    A **mean of the per-task medians**: `cost_accuracy` has already collapsed
    each task's seeds with `seed_center` (`SEED_STATISTIC`, the median, because
    one arm's seed population is bimodal), and this collapses across tasks with
    `task_center` (`TASK_STATISTIC`, the mean, because every task is a real
    observation and the extremes are the informative ones). The two steps answer
    two different questions and no longer take the same statistic; read both
    constants before quoting a cross.

    On a grouped frame (`group_tasks`, `TASK_GROUPING`) the rows are data sets
    and `n` counts data sets: the per-task medians have been collapsed once more,
    within each data set, before this step sees them.

    Worth knowing when reading it off a *log* cost axis: an arithmetic mean of
    ratios is **not** at the visual midpoint of the markers it summarises, since
    the axis is logarithmic and the mean is not. A cross that looks high against
    its cloud on the cost axis is that, not a bug.
    """
    return (frame.groupby(["family", "rank"], as_index=False)
                 .agg(x=("x", task_center), y=("y", task_center),
                      n=("task", "nunique")))


def task_statistic_table(frames):
    """What `TASK_STATISTIC` is worth: mean against median, per metric and arm.

    The one number a reader of section 3 has to be able to check for himself,
    because the choice of statistic changes a *ranking* on one of the six
    metrics -- CRPS, where `SCORE` is first of the Gaussian arms as a mean and
    third as a median -- and nothing on the figure says which one was used.
    Takes the frames the grid returns, so the columns are already in the unit
    the panels are drawn in (per cent, or nats per coordinate on the log score).
    """
    rows = []
    for metric, frame in frames.items():
        for (fam, rank), sub in frame.groupby(["family", "rank"], sort=False):
            rows.append(dict(
                metric=GRID_TITLE.get(metric, METRICS[metric]["label"]),
                arm=arm_label(fam, rank, tex=False),
                n=sub["task"].nunique(),
                mean=task_center(sub["y"], how="mean"),
                median=task_center(sub["y"], how="median")))
    out = pd.DataFrame(rows)
    out["shift"] = out["median"] - out["mean"]
    return out.round(3)


def drawn_x(frame, mode):
    """The cost positions a panel actually puts a glyph at, in `mode`.

    The tick ladder is sized off this rather than off the frame, because the
    two differ: the main figure draws crosses at the arm centres, which span
    0.96x to 1.82x, while the per-task values behind them reach 3.15x. Sizing
    its axis to the points it does not draw would leave a third of every panel
    empty, and a fixed tick ladder pulls the view out to its last tick.
    """
    return arm_centers(frame)["x"] if mode == "mean" else frame["x"]


#: Round multiples for the cost axis, coarse first. `1.5` is in the second
#: ladder only: on a panel a fifth of a text width wide it overprints `2x`, so
#: it is used only where the coarse ladder would leave a single labelled tick
#: -- which is what happens on the main figure, whose crosses span less than
#: an octave.
COST_TICKS = (1, 2, 3, 4, 6, 8, 12, 16)
COST_TICKS_FINE = (1, 1.5, 2, 3, 4, 6, 8, 12, 16)

#: Two last resorts, coarse first, for an axis that neither ladder above gives
#: two ticks on: the per-data-set figures of section 3.3, where ERA5 spans
#: roughly 0.87x to 1.22x and EUPPBench 0.85x to 1.00x. Only reached below 1.5x,
#: so `1.25` has no `1.5` to collide with; `0.9` is in the second one only,
#: because next to `1x` it touches unless the axis is as narrow as EUPPBench's.
COST_TICKS_SUB = (0.5, 0.75, 1, 1.25)
COST_TICKS_SUB_FINE = (0.8, 0.9, 1, 1.1, 1.2)


def cost_ticks(x, min_ticks=2):
    """The ladder for a cost axis covering `x`, as multiples of the reference."""
    lo, hi = float(np.min(x)) / 1.05, float(np.max(x)) * 1.05
    coarse = [t for t in COST_TICKS if lo <= t <= hi]
    if len(coarse) >= min_ticks:
        return coarse
    fine = [t for t in COST_TICKS_FINE if lo <= t <= hi]
    if len(fine) >= min_ticks:
        return fine
    for ladder in (COST_TICKS_SUB, COST_TICKS_SUB_FINE):
        sub = [t for t in ladder if lo <= t <= hi]
        if len(sub) >= min_ticks:
            return sub
    return fine or [1]


def draw_arm(ax, sub, color, mean_min_n=2, zorder=4, mode="both", alpha=1.0,
             size=22):
    """One arm: a marker per task, a cross at the mean over them, or both.

    `mode` picks which of the two glyphs the panel carries, and it is the only
    difference between the paper's two cost-accuracy figures:

    * `"mean"` -- the cross alone, one glyph per arm. The main-paper figure.
      (The name is the *mode*, kept for the API; what the cross sits at is
      `TASK_STATISTIC`, which is the median.)
      Six arms over up to six tasks put thirty markers on a panel a third of a
      column wide; the clouds overlap into a smear, and what the main text
      compares is where each arm sits on average, not how it scatters.
    * `"points"` -- the markers alone, one per task, no summary. The appendix
      figure, where the spread over tasks *is* the subject and a cross on top
      of it only hides points.
    * `"both"` -- the two together, which is what a single panel drawn on its
      own shows.

    No box, no whiskers, no shading. With two to six tasks per arm there are no
    quartiles worth drawing, and a rectangle over two numbers states a spread it
    has not measured; the markers *are* the distribution at this sample size,
    and the cross is the only summary the data supports.

    A marker is hollow when the run behind it was fitted with an objective other
    than the one its timing was measured under (`strict=False` only), so a
    point that is not loss-matched is visible as one. There is no hollow form of
    the cross: a mean over a mixture of objectives is one number and cannot say
    which of its terms was mismatched, so `mode="mean"` is drawn filled and the
    mismatch stays a fact about the points, reported by `cost_accuracy`.

    Returns True when a cross was drawn, so the caller can key it.
    """
    if mode not in ("both", "points", "mean"):
        raise ValueError(f"mode must be 'both', 'points' or 'mean', got {mode!r}")
    if mode != "mean":
        matched = sub["objective_matched"].to_numpy(dtype=bool)
        for mask, fill in ((matched, color), (~matched, "none")):
            if mask.any():
                ax.scatter(sub["x"][mask], sub["y"][mask], marker=POINT_MARKER,
                           s=size, facecolor=fill, edgecolor=color, linewidth=0.7,
                           alpha=alpha, zorder=zorder)
    if mode != "points" and len(sub) >= mean_min_n:
        ax.plot(task_center(sub["x"]), task_center(sub["y"]),
                marker=CENTER_MARKER,
                color=color, ms=8, mew=CFG.line_width, linestyle="none",
                zorder=zorder + 1)
        return True
    return False

In [ ]:
#: The stage-1-only costing of SCORE, drawn in the second cost-accuracy grid
#: (`plot_cost_accuracy_grid(include_stage1=True)`): the same task and the same
#: final score as the regular SCORE point, but costed at stage 1 alone rather
#: than the full two-stage fit.
def stage1_seed_ms(rows=None):
    """Stage-1's own `step_ms_median`, per (task, seed), for the `ours` arm.

    `seed_costs` already collapsed a staged arm's two benchmark rows into one
    measurement keyed on `effective_step_ms` -- the epoch-weighted mean over
    both stages, which is what a full training run actually pays -- and does
    not keep the per-stage number around. This reads it straight from
    `RUNTIME_ROWS` instead of going back through that collapse.
    """
    rows = RUNTIME_ROWS if rows is None else rows
    seen = {}
    for r in rows:
        if not (r.get("staged") and r.get("stage") == 1):
            continue
        if family(r["method"]) != "ours":
            continue
        if r.get("status") not in mrt.MEASURED_STATUS:
            continue
        try:
            task = mrt.identity(r)
        except ValueError:               # a datamodule the mapping has not met
            continue
        seen[(task, r.get("seed"))] = float(r["step_ms_median"])
    return seen


def stage1_only_frame(frame):
    """`frame`'s `ours` rows, recosted at stage 1 alone.

    Same task, same score `y` -- the metric a full two-stage run reports --
    only `x` changes, to what the campaign would have paid stopping after stage
    1, collapsed over seeds with `seed_center` like every other cost on the
    panel. Stage 1 is the heavier of the two stages everywhere it is measured (the
    spectral core; stage 2 is a cheap calibration pass), so its own per-step
    cost is *larger* than `ours`'s blended `effective_step_ms`, and this point
    sits to the *right* of the regular `ours` marker, not the left.

    Not part of `cost_accuracy`: it does not correspond to a run that was ever
    scored on its own, so it has no place among the cells `coverage` counts. It
    exists only to be drawn.
    """
    ms_by_task_seed = stage1_seed_ms()
    ours = frame[frame["family"] == "ours"]
    if ours.empty or not ms_by_task_seed:
        return ours.iloc[0:0].copy()
    # A plain dict, not a `task`-indexed Series: `task` is itself a tuple, and
    # `.loc[a_tuple]` on a Series reads the tuple as a multi-axis indexer
    # rather than as one key.
    diag_by_task = dict(COSTS.groupby("task")["diag_s_per_step"].first().items())
    rows = []
    for _, row in ours.iterrows():
        seeds_ms = [ms for (t, _), ms in ms_by_task_seed.items() if t == row["task"]]
        if not seeds_ms or row["task"] not in diag_by_task:
            continue
        new = row.copy()
        new["family"] = "ours_stage1"
        new["x"] = (seed_center(seeds_ms) / 1e3) / diag_by_task[row["task"]]
        rows.append(new)
    return pd.DataFrame(rows) if rows else ours.iloc[0:0].copy()


In [ ]:
def plot_cost_accuracy(metric="crps", strict=True, ax=None, legend=True,
                       annotate=True, save_as=None, verbose=True,
                       y_transform=None, mode="both", per_seed=False,
                       point_alpha=1.0, point_size=22, grouping="default",
                       tasks=None):
    """The panel: cost against relative score, one marker per (arm, task).

    `strict` keeps only cells whose synced run was fitted with the objective its
    timing was measured under (see :func:`match_arm`); `strict=False` admits the
    rest and draws them hollow, so a mismatched point is visible as one.

    `mode` is what each arm draws -- one marker per task, a cross at their mean,
    or both (see :func:`draw_arm`). It also decides the glyph the legend keys
    the arms with, so the key always shows what is on the panel.

    The panel states its own coverage: the legend carries the number of tasks
    behind each arm, and a note appears whenever the arms do not rest on the
    same set of tasks -- markers over different tasks are not a comparison, and
    that has to be readable off the saved PDF rather than out of this docstring.

    `annotate=False` suppresses that note, for a caller that draws it once for a
    whole grid instead of once per panel. It also keeps the note's
    y-limit expansion out of a shared axis, where it would apply to every panel.

    `y_transform`, when given, is applied to the frame's `y` column -- and to
    the horizontal guide line at the same point -- before anything is drawn or
    limited, so everything downstream (markers, mean cross, margins) works in
    the transformed unit. `plot_cost_accuracy_grid` uses it to plot a signed
    percentage improvement instead of the raw ratio; called on its own here the
    default `None` leaves the ratio scale (and this function's own y-label)
    untouched.

    `tasks`, when given, restricts the panel to those task identities (the
    per-data-set figures of section 3.3), and `grouping` overrides
    `TASK_GROUPING` for this panel only -- `None` draws one marker per task.
    """
    spec = METRICS[metric]
    costs = (None if tasks is None
             else COSTS[COSTS["task"].map(lambda t: t in tasks)])
    frame, dropped = cost_accuracy(metric, strict=strict, verbose=verbose,
                                   per_seed=per_seed, costs=costs)
    if frame.empty:
        raise ValueError(f"no (task, arm) cell has both a timing and a "
                         f"{metric} score" + ("" if not strict else
                         "; try strict=False"))
    # One row per data set from here on (`TASK_GROUPING`): the markers, the
    # cross and the key's counts are all taken over groups, so the cross stays
    # the statistic of exactly the markers the appendix draws.
    frame = group_tasks(frame, grouping)
    if y_transform is not None:
        frame = frame.copy()
        frame["y"] = y_transform(frame["y"].to_numpy())
    fig = None
    if ax is None:
        # Taller than the 0.78 a bare panel would take: the legend strip below
        # the axes costs height, and this keeps the data area square-ish.
        fig, ax = pc.subplots(1, 1, aspect=0.95, columns="single")

    # The two reference lines are guides, not data: dashed, thin and behind
    # everything, or they read as the strongest features on the panel.
    guide = dict(color=pc.COLORS["guide"], lw=CFG.thin_line_width * 0.8,
                 linestyle=(0, (4, 3)), alpha=0.8, zorder=0)
    base = 1.0 if spec["scale"] == "ratio" else 0.0
    if y_transform is not None:
        base = float(y_transform(np.array([base]))[0])
    ax.axvline(1.0, **guide)
    ax.axhline(base, **guide)

    arms = sorted({(f, r) for f, r in zip(frame["family"], frame["rank"])},
                  key=arm_order)
    # A cross needs two tasks to be a summary rather than a restatement of a
    # single point -- except when it is the *only* glyph the panel draws, where
    # holding it back would leave an arm in the key and nowhere on the panel.
    # What the cross of a one-task arm means is then the same thing the "arms
    # rest on different task sets" note already warns about.
    min_n = 1 if mode == "mean" else 2
    task_sets, handles, any_mean = {}, [], False
    for fam, rank in arms:
        sub = frame[(frame["family"] == fam) & (frame["rank"] == rank)]
        color = arm_color(fam, rank)
        task_sets[(fam, rank)] = frozenset(sub["task"])
        any_mean |= draw_arm(ax, sub, color, mean_min_n=min_n, mode=mode,
                             alpha=point_alpha, size=point_size)
        # `ref.` rather than `(reference)`: the legend sits under a 3.25 in
        # panel and every character of a label is width the panel does not have.
        # The reference arm is the denominator of *both* axes, so every one of
        # its markers lands exactly on the crossing of the two guides -- which
        # is worth saying in the key, because a stack of markers on the crossing
        # otherwise looks like a coincidence rather than the construction.
        name = arm_label(fam, rank, tex=False)
        if fam == spec["reference"]:
            name += ", ref."
        handles.append((f"{name} ({len(sub)})", color, len(sub)))

    # A log cost axis. The arms span 1.0x to 5.1x, and on a linear axis the
    # 1.0-1.6x band -- where Diag, SCORE and the whole station-graph task live --
    # collapses into the left margin.
    ax.set_xscale("log")
    # A sparse ladder rather than matplotlib's decades: the whole panel lives
    # inside one decade, and 1.25 next to 1.5 collides at this font size. Sized
    # to the glyphs this `mode` draws, not to the frame -- a fixed ladder pulls
    # the view out to its last tick, so a ladder built for points it does not
    # draw would leave the panel two thirds full.
    ticks = cost_ticks(drawn_x(frame, mode))
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{t:g}" + r"$\times$" for t in ticks])
    ax.set_xticks([], minor=True)
    ax.margins(x=0.09, y=0.10)
    # Both labels name `spec["reference"]` rather than spelling the arm out, so
    # a change of denominator cannot leave a panel captioned with the old one.
    ref_name = arm_label(spec["reference"], 0, False)
    ax.set_xlabel(rf"per-step cost vs.\ {ref_name}")
    ax.set_ylabel(f"{spec['label']} / {ref_name}" if spec["scale"] == "ratio"
                  else rf"$\Delta$ {spec['label']} vs.\ {ref_name}")

    if legend:
        style = (dict(marker=CENTER_MARKER, ms=6, mew=CFG.thin_line_width)
                 if mode == "mean" else dict(marker=POINT_MARKER, ms=4))
        proxies = [plt.Line2D([], [], color=c, linestyle="none", **style)
                   for _, c, _ in handles]
        labels = [name for name, _, _ in handles]
        if any_mean and mode == "both":
            # The cross is a second glyph on the panel, so it gets a key rather
            # than a sentence in a caption nobody reads next to the figure. The
            # label names the statistic rather than saying "summary": a cross
            # over a cloud is read as a mean unless it says otherwise. Not
            # needed when the cross is the only glyph: there the arms' own keys
            # are crosses and a seventh entry would say what they already do.
            # Near-black, not the mid grey it would naturally take: every arm
            # draws its cross in its own colour, and the diagonal arm's colour
            # *is* mid grey -- a grey key would read as that arm's entry.
            proxies.append(plt.Line2D([], [], color="0.1", marker=CENTER_MARKER,
                                      linestyle="none", ms=6,
                                      mew=CFG.thin_line_width))
            labels.append(f"{TASK_STATISTIC} over "
                          + ("tasks" if TASK_GROUPING is None else "data sets"))
        # Outside, under the axes, when this function owns the figure. Seven
        # entries do not fit inside a single-column panel without landing on
        # data, and which corner is free changes from metric to metric -- on the
        # variogram panel `loc="best"` put the box straight through the diagonal
        # arm's mean. A caller that passes its own `ax` gets an inside legend,
        # since it owns the figure's layout.
        kw = dict(fontsize=CFG.legend_size * 0.85, handletextpad=0.3,
                  borderpad=0.35, labelspacing=0.25, columnspacing=1.0,
                  handlelength=1.0)
        if fig is not None:
            fig.legend(proxies, labels, loc="outside lower center", ncol=3,
                       frameon=False, **kw)
        else:
            ax.legend(proxies, labels, loc="best", **kw)

    notes = []
    drawn = {k: v for k, v in task_sets.items() if k[0] != spec["reference"]}
    if len(set(drawn.values())) > 1:
        notes.append("arms rest on different task sets")
    if not strict and not frame["objective_matched"].all():
        notes.append("hollow: objective differs from the timed one")
    if notes and annotate:
        # Top left, in a strip opened above the data: the interesting corner of
        # this panel is the bottom left (cheap and accurate) and the reference
        # arm sits at the top left, so the note needs room of its own rather
        # than a corner it can share.
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + (0.09 * len(notes) + 0.04) * (hi - lo))
        ax.text(0.025, 0.975, "\n".join(notes), transform=ax.transAxes,
                fontsize=CFG.annotation_size, color="0.35",
                ha="left", va="top", linespacing=1.35, zorder=8,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8,
                          pad=1.5))
    if save_as and fig is not None:
        pc.save(fig, save_as)
    return fig, ax, frame

### 3.1 The main-paper figure (Figure 4)

The six metrics on one grid: **MSE, ES (energy score) and KS (kernel score)**
across the top, **CRPS, VS (variogram score) and NLL (the log score)** below,
with the key in a strip beneath. Reading them side by side is what separates
"this arm forecasts better" from "this arm models the *dependence* better" —
MSE and CRPS see only the marginals (MSE only the mean), while the energy,
kernel, variogram and log scores are the ones a covariance can actually move.

Five panels quote their metric as a percentage improvement over the same
`phys_diag` reference on the same per-step cost axis — positive is better, 0 %
is the reference itself. The cost axis is shared by all six; the y-axis is
not: **every panel has its own scale and its own ticks**. A shared range would
let SB's energy score set the range of MSE and KS as well, and the Gaussian arms
-- under $1.5$ points apart on that row -- would sit flat on the guide line. The
price is that heights are not comparable *between* panels, only positions
within one; the tick labels carry the magnitudes.

**The log score is the sixth panel and the one exception, in nats per
coordinate rather than per cent.** It cannot be a percentage. Its
reference differs in sign and size from task to task — negative on ERA5 and
depth, $+1.75$ on EUPPBench — so SCORE's gain of $1.12$ nats on ETTh1
univariate $T{=}96$ and $0.25$ on EUPPBench would read as percentages of very
different denominators: a statement about the reference, not the arm. Depth's reference
is $-1.69$ nats per coordinate, genuinely negative, so a quotient would mirror
that whole panel row and "closer to the reference" would read as "further from
it" — which is exactly the case `cost_accuracy` refuses for the ratio metrics
and the log score never had to. What is drawn instead is the **difference in
nats per coordinate, sign-flipped so that up is better** like every other panel,
which is not a fallback but the natural scale-free form for a density: it is a
log likelihood ratio, so SCORE at $+1.76$ nats on ERA5 means
$e^{1.76} \approx 5.8\times$ the per-coordinate predictive density of the
diagonal, and it is invariant to the units the data are in for exactly the
reason a percentage is — a change of units shifts every arm's log score by the
same constant and cancels in the difference. **The quadrant therefore reads the
same way on all six panels** (left of the vertical guide is cheaper, above the
horizontal guide is better). Its ticks are on the **left**, like every other
panel's. It has numbers and no
name: the unit belongs in the figure's caption, which should say *NLL panel in
nats per coordinate saved against the diagonal, positive is better* — at $0.85$
in wide the panel has no room for a second axis title.

That panel is also the only one an arm is *absent* from: SB has no log
score at all, because a sample-based forecast has no density (`nll` is null in
any of its `metrics.json`). Its key is still on the figure — it is on the
other five panels — so the log-score panel is the one place a colour in the key
has no marker under it.

⚠️ **It is also the panel that needs care.** SCORE
leads on **every task except depth, the two multivariate ETTm2 tasks, the two
weather tasks, the two electricity tasks and the two traffic tasks** -- from
$+0.25$ nats on EUPPBench to $+1.76$ on ERA5 over the other eighteen -- and those
nine contribute $-60.9$ (depth, from a kernel-score
fit whose five seeds report $7.1$ to $17\,974$ nats/dim where the diagonal is flat
at $-1.69$), $-195$ and $-3.3 \times 10^{6}$ (ETTm2), and $-61.1$ and $-69.9$
(weather, where every kernel-score fit, the diagonal reference included, sits at
$+230$ nats/dim or more), and $-8.1 \times 10^{3}$ and $-2.3 \times 10^{4}$
(electricity, where SCORE diverges on every seed), and $-0.50$ and $-0.56$
(traffic, finite and close: SCORE at $+1.3$ to $+1.4$ nats/dim, the kernel-score
diagonal at $+0.84$ to $+0.87$). As data sets SCORE leads on six of the eleven,
and depth ($-60.9$), weather ($-65.5$), ETTm2 ($-96.5$, the median of two sane
and two degenerate cells), electricity ($-1.5 \times 10^{4}$) and traffic
($-0.53$) are the five below. With `TASK_STATISTIC = "median"` the cross is
$+0.25$ -- the sixth of eleven, EUPPBench's own value -- and the panel reads as
the six do; as a mean it would be $-1421$. The appendix figure has no such protection: it draws those four dots,
down to $-1.5 \times 10^{4}$ with electricity's SCORE, so its NLL panel is ruled
by them until one of the ways out in section 3 is taken. That is a real
property of fitting a $V = 238\,560$ covariance to a score that does not
constrain the density — the NLL-fitted sibling is stable at $-4.78$ and beats the
NLL-fitted diagonal — but it is not a number that belongs in a cross summarising
"the log score of SCORE". The coverage note in section 3 sets out the ways
to handle it; the caption should say which one the figure uses.

**Each arm is one cross**, at the `TASK_STATISTIC` (currently the median) of the
per-data-set values behind it; a data set's value is the `GROUP_STATISTIC`
(median) of its tasks, and a task's the **median** of its seeds
(`SEED_STATISTIC`). The steps want robustness for different reasons: within a
task, one arm's seed population can be bimodal (SB collapses on 2 of 5
seeds at illness $T = 36$ and on all five at depth, on all twenty at ETTm
univariate); across tasks, the set is a convenience sample of twenty-seven,
twenty-four of them time series and sixteen of those ETT, which is why the cross
is taken over the eleven data sets instead (`TASK_GROUPING`).

**What the choice costs.** `task_statistic_table(GRID_FRAMES)` prints every
cross both ways, over data sets. The gaps are led by the **log score, where
SCORE is $-1421$ as a mean and $+0.25$ as a median**, decided entirely by whether
depth, ETTm2, weather and electricity are averaged or ranked (LorD $r{=}1$'s NLL
cross is $+5.0$ against $+0.10$ and LorD $r{=}16$'s $+0.65$ against $+0.05$); then
SB on ES ($-8.7$ mean against $-0.8$ median), CRPS ($-6.3$ against
$+0.6$) and VS ($-1.6$ against $+1.9$, see section 3) -- with eleven data sets
each median is the sixth value, which on CRPS and ES falls on the healthy side
of the gap between SB's five collapsed or over-dispersed data sets and
its six others. SCORE on **CRPS** is
$+6.3\,\%$ as a mean, lifted by traffic's $+26.2\,\%$, and $+3.4\,\%$ as a
median — ahead of LorD $r{=}16$ ($+2.1$ / $+2.0$) either way. As a median it is first
of the four Gaussian arms on all five percentage panels, MSE included ($+0.51$
against LorD $r{=}16$'s $+0.40$); as a mean it is first on MSE, CRPS and VS and third
on KS ($+0.56$, behind LorD $r{=}1$'s $+0.74$ and LorD $r{=}16$'s $+0.57$) and on ES
($+0.74$, behind LorD $r{=}16$'s $+0.86$ and LorD $r{=}1$'s $+0.80$).

All five arms rest on the same eleven data sets, VS included, so crosses
beside each other compare like with like. That is what `COVERAGE` above is for; the counts are
deliberately not in the key.

SCORE (stage 1) -- the same runs costed at stage 1 alone rather than at the
epoch-weighted blend of both stages -- is **not drawn**: at $1.08\times$ against
SCORE's $1.06\times$ as a data-set median it sits on top of the SCORE cross on
every panel and only made the key harder to read.
`plot_cost_accuracy_grid(include_stage1=True)` still draws it.

In [ ]:
from matplotlib.layout_engine import PlaceHolderLayoutEngine

#: The six metrics of the joint figures, in reading order: MSE, ES (energy
#: score) and KS (kernel score) across the top, CRPS, VS (variogram score) and
#: the log score below. MSE leads its row because it is the only one of the six
#: that does not see the covariance at all -- an arm can move on it only by
#: moving its mean; CRPS leads the second row because every drawn arm reports
#: it, and the log score closes it because it is the one panel with its own
#: unit and the one an arm can be absent from.
#:
#: The split between the rows is by size of the effect: MSE, ES and KS all
#: move by a few per cent at most, CRPS, VS and the log score by much more.
#:
#: **The log score is the one panel of the six that is not a percentage.** It
#: cannot be one. It is the only metric with `scale="difference"`, because it is
#: not positive, and a percentage needs a denominator: the `phys_diag` reference
#: is 0.12 nats per coordinate on ETTh1 univariate T=96 and 1.75 on EUPPBench, so
#: SCORE's similar 1.1-1.3 nat gain on the two reads as +976 % on the first and
#: +14 % on the second. That is a statement about the reference, not about the
#: method, and on a task whose reference straddled zero it would change sign.
#:
#: What it is instead is a **difference in nats per coordinate, sign-flipped so
#: that positive is better** like every other panel. That is not a fallback: a
#: difference of log densities is already the scale-free quantity a percentage
#: is trying to construct. It is a log likelihood ratio per coordinate -- SCORE's
#: +1.20 nats means e^1.20 = 3.3x the per-coordinate predictive density of the
#: diagonal -- and, exactly like the percentages, it is invariant to the units
#: the data happen to be in, because a change of units shifts every arm's log
#: score by the same constant and cancels in the difference. So the panel is
#: directly comparable *across tasks*, which is all the normalization is for;
#: only its *height* is not comparable to the height of a percentage panel,
#: which is why it carries its own axis. That axis is drawn with numbers and no
#: name -- `nats/dim, positive is better` is a caption's job, not a 0.85 in
#: panel's.
#:
#: The quadrant still reads the same way on it -- left of the vertical guide is
#: cheaper, above the horizontal guide is better than the diagonal -- because
#: the quadrant depends on the *signs*, and both survive the change of unit.
#:
#: One arm is absent from it: SB (`generative`) has no log score at all (a
#: sample-based forecast has no density; `nll` is null in all 20 of its
#: `metrics.json`). Its key is still on the figure, because it is on the other
#: five panels, so the log-score panel is the one place a colour in the key has
#: no marker under it.
GRID_METRICS = ("mse", "energy_score", "kernel_score", "crps",
                "variogram_score", "nll_per_d")

#: The main-paper figure's layout. The metrics fill it row-major, so the two
#: numbers here decide both the shape of the figure and which row a metric
#: lands in. With the log score on the grid the six metrics fill all six slots
#: and there is none left for the key, which is why both figures pass
#: `legend="row"` and put it in a strip underneath.
GRID_SHAPE = (2, 3)

#: The appendix figure's layout: **the six metrics in one line**. Six
#: panels fill the row, so there is no spare slot for the key and it goes in a
#: strip below instead (`legend="row"`), on one line. A line only works at the
#: full text width -- six panels across a single column would be 0.54 in each
#: -- so its caller pairs this with `columns="double"`. This layout runs
#: `sharey=False`, so every panel carries its own numbers on its own left, as
#: in the main figure.
APPENDIX_SHAPE = (1, 6)

#: The appendix figure's metric order: the marginal scores first (MSE, CRPS),
#: then the multivariate ones (ES, VS, KS), and the log score last so its
#: separate unit stays at the end of the line.
APPENDIX_METRICS = ("mse", "crps", "energy_score", "variogram_score",
                    "kernel_score", "nll_per_d")

#: Panel-title shorthand for the joint figures only. `METRICS[m]["label"]`
#: still spells the score out everywhere else (the coverage table), where the
#: full name has room; here five titles have to sit over panels a fifth of a
#: text width wide.
GRID_TITLE = {"energy_score": "ES", "variogram_score": "VS",
              "kernel_score": "KS", "nll_per_d": "NLL"}


#: How a metric's normalized value becomes a signed improvement, keyed on its
#: `scale`. **Positive is better in both**, which is what lets the horizontal
#: guide mean one thing across a grid whose panels carry two units: a marker
#: above it beats the reference, whatever the panel measures.
IMPROVEMENT = {
    "ratio": lambda y: 100.0 * (1.0 - np.asarray(y, dtype=float)),
    "difference": lambda y: -np.asarray(y, dtype=float),
}

#: The y-axis name per `scale`, for the **margin** label only -- the one that
#: names the unit the majority of the panels carry. A panel whose unit is in
#: the minority (the log score, on a grid of percentages) is deliberately left
#: *unlabelled*: at 0.85 in wide there is no room for a second axis name that
#: is not either abbreviated past usefulness or long enough to steal the
#: column's width, and the figure's caption is the right place for a unit that
#: applies to one panel. So the log score's numbers are on the page and
#: `nats/dim, positive is better` belongs in the caption text. The percentage
#: label carries no unit either: "per cent over the diagonal" is the caption's too.
IMPROVEMENT_LABEL = {"ratio": "Improvement",
                     "difference": r"Log score saved [nats/dim]"}

#: Legend order, distinct from `arm_order` (which orders the cost tables of
#: section 2 by increasing generality). Grouped rather than graded: Diag (the
#: reference) with SB, then the two LorD ranks, then the two SCORE costings, so
#: the key reads as three pairs down the main figure's column and along the
#: appendix figure's row.
GRID_LEGEND_ORDER = ["phys_diag", "generative", "phys_lora", "ours", "ours_stage1"]


def grid_legend_order(arm):
    fam, rank = arm
    idx = (GRID_LEGEND_ORDER.index(fam) if fam in GRID_LEGEND_ORDER
           else len(GRID_LEGEND_ORDER))
    return (idx, int(rank or 0))


def _legend_proxies(arms, reference, mode):
    """Keys for the drawn arms, in the glyph the panels actually carry.

    The marker follows `mode`: a cross where the panels draw crosses, a dot
    where they draw dots. A key whose glyph is not the one on the panel is
    worse than no key at all -- the reader matches shapes first and colours
    second, and the main figure has nothing but crosses on it.
    """
    order = sorted(arms, key=grid_legend_order)
    style = (dict(marker=CENTER_MARKER, ms=6, mew=CFG.thin_line_width)
             if mode == "mean" else dict(marker=POINT_MARKER, ms=4))
    proxies = [plt.Line2D([], [], color=arm_color(fam, rank), linestyle="none",
                          **style) for fam, rank in order]
    labels = [arm_label(fam, rank, tex=False)
              + (", ref." if fam == reference else "") for fam, rank in order]
    return proxies, labels


def _link_y(axes, hide_inner=True):
    """Put `axes` on one y-axis after the fact: common limits, one tick column.

    `sharey` at construction time cannot express "share with the panels in this
    row that carry the same unit" -- `plt.subplots` resolves it before a single
    metric has been placed -- so a mixed-unit grid is built unshared and the
    panels that belong together are linked here. Limits and `labelleft` rather
    than `Axes.sharey`, which raises once a panel has been drawn into.
    """
    if len(axes) < 2:
        return
    lo = min(ax.get_ylim()[0] for ax in axes)
    hi = max(ax.get_ylim()[1] for ax in axes)
    for k, ax in enumerate(axes):
        ax.set_ylim(lo, hi)
        if hide_inner and k:
            ax.tick_params(labelleft=False)


def _carve_footer(fig, inches):
    """Grow the canvas at the bottom and keep the layout engine off the strip.

    The axis label and (in the line layout) the key are placed by hand rather
    than negotiated by `fig.supxlabel` and an `outside` legend, so that the
    label can sit right under the tick labels it belongs to and the key below
    it. `inches` is the height they need together; growing the canvas by
    exactly that and then confining constrained layout to the region above it
    leaves the strip untouched by the panels or the titles.

    Returns the strip's height as a fraction of the grown figure -- the y that
    the bottom of the panels' region now sits at.
    """
    fig_w, fig_h = fig.get_size_inches()
    fig.set_size_inches(fig_w, fig_h + inches)
    frac = inches / (fig_h + inches)
    # `rect` is (left, bottom, WIDTH, HEIGHT) -- not (left, bottom, right,
    # top). Passing 1 for the last element (rather than 1 - bottom) asks
    # constrained layout for a region that runs past the top of the canvas,
    # which pushes everything up and off it instead of leaving the strip at
    # the bottom.
    fig.get_layout_engine().set(rect=(0, frac, 1, 1 - frac))
    return frac


def _axes_center_x(fig, axs):
    """Centre of the axes grid in figure coordinates.

    What the label and the key are centred on, rather than the canvas: a plain
    figure-fraction centre (x=0.5) would sit under the y-axis label strip on
    the left rather than under the middle column. Draws first, because the
    positions are not final until constrained layout has run.
    """
    fig.canvas.draw()
    xs = [b for ax in axs for b in (ax.get_position().x0, ax.get_position().x1)]
    return (min(xs) + max(xs)) / 2


def _side_boxes(ax, renderer):
    """The panel's frame, and one box per label it draws around it, in pixels.

    Tick labels of both axes (only those of ticks inside the view, the ones
    that are drawn), the title and the offset texts. One box *per label*
    rather than `get_tightbbox`'s union, because the union is exactly what
    over-reserves: an x tick label below the frame and the y tick labels
    beside it become one rectangle that neither of them fills.
    """
    frame = ax.get_window_extent(renderer)
    boxes = []
    for labels, along in ((ax.get_xticklabels(), "x"), (ax.get_yticklabels(), "y")):
        lo, hi = (frame.x0, frame.x1) if along == "x" else (frame.y0, frame.y1)
        for lab in labels:
            if not (lab.get_visible() and lab.get_text()):
                continue
            box = lab.get_window_extent(renderer)
            mid = (box.x0 + box.x1) / 2 if along == "x" else (box.y0 + box.y1) / 2
            if lo - 1 <= mid <= hi + 1:
                boxes.append(box)
    for text in (ax.title, ax.xaxis.get_offset_text(), ax.yaxis.get_offset_text()):
        if text.get_visible() and text.get_text():
            boxes.append(text.get_window_extent(renderer))
    return frame, boxes


def _compact_columns(fig, grid, clearance):
    """Close the gaps between the columns down to what their labels need.

    Constrained layout reserves one margin per side of a column, the widest
    over all the rows, and puts the two margins of a gap side by side. On the
    main figure that is the `1.5x` hanging off the right of every bottom
    panel *plus* the widest y tick labels of the next column -- but the two
    never meet: the x label sits under its row's frames and the y labels
    within their height. So this measures, row by row, how far the labels of
    two neighbouring panels reach towards each other, keeps every pair that
    shares a height `clearance` inches apart (a frame counts as a label), and
    hands the width it frees to the panels. The outer edges stay where the
    layout put them.

    Freezes the layout afterwards, since the next draw would otherwise run
    constrained layout again and undo it.
    """
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    pad = clearance * fig.dpi
    n_rows, n_cols = grid.shape
    sides = [[_side_boxes(ax, renderer) if ax.axison else None for ax in row]
             for row in grid]
    gaps = []
    for k in range(n_cols - 1):
        need = pad
        for r in range(n_rows):
            if sides[r][k] is None or sides[r][k + 1] is None:
                continue
            (frame_a, boxes_a), (frame_b, boxes_b) = sides[r][k], sides[r][k + 1]
            for a in boxes_a + [frame_a]:
                for b in boxes_b + [frame_b]:
                    if a.y0 < b.y1 and b.y0 < a.y1:
                        need = max(need, (a.x1 - frame_a.x1)
                                   + (frame_b.x0 - b.x0) + pad)
        gaps.append(need)
    frames = [[s[0] for s in col if s is not None] for col in zip(*sides)]
    left = min(f.x0 for f in frames[0])
    right = max(f.x1 for f in frames[-1])
    width = (right - left - sum(gaps)) / n_cols
    fig_w = fig.bbox.width
    x0 = left
    for k in range(n_cols):
        for ax in grid[:, k]:
            pos = ax.get_position()
            ax.set_position([x0 / fig_w, pos.y0, width / fig_w, pos.height])
        x0 += width + (gaps[k] if k < n_cols - 1 else 0)
    fig.set_layout_engine(PlaceHolderLayoutEngine(True, True))


def plot_cost_accuracy_grid(metrics=GRID_METRICS, strict=True, columns="single",
                            aspect=0.68, save_as=None, verbose=False,
                            include_stage1=False, shape=GRID_SHAPE,
                            mode="both", legend="slot", sharey="row",
                            legend_ncol=None, per_seed=False, point_alpha=1.0,
                            point_size=22, grouping="default", tasks=None,
                            title=None, y_nbins=6, col_gap=None):
    """One panel per metric on shared axes, in one of the paper's two layouts.

    The figures differ in what each arm draws and in where the panels and the
    key sit; everything else -- the cells, the axes, the normalizations -- is
    the same code, so the appendix figure cannot drift away from the main one.

    * **main paper**: `shape=GRID_SHAPE`, `mode="mean"`, `legend="row"`,
      `legend_ncol=3`, `sharey=False`. Two rows of three, a cross per arm, and
      the key in two rows of three beneath.
    * **appendix**: `shape=APPENDIX_SHAPE`, `mode="points"`, `legend="row"`,
      `columns="double"`, `sharey=False`. One line of six at the full text
      width, a dot per task, and the key in a single row beneath.

    `sharey="row"` links the panels of a row to one y-axis (the main figure's
    until 2026-10-01, with the log score's numbers on the right margin);
    `sharey=False` gives every panel its own, which both figures now use --
    MSE moves by a few per cent and the sampler's ES by ten, and one axis over
    both flattens the first into the guide line.

    **A mixed-unit grid is linked by hand.** `sharey` is a `plt.subplots`
    argument, resolved before any metric is placed, so it cannot know that the
    log-score panel is in nats while its row-mates are in per cent. When the
    metrics do not all share a `scale`, the figure is built unshared and
    `_link_y` links the same-unit panels of each row afterwards; the odd panel
    then names its own y-axis, and `fig.supylabel` carries the majority unit.

    `legend_ncol` caps the width of the `legend="row"` key, and the strip grows
    to hold the rows that implies. Six keys fit on one line at the full text
    width and not at a single column, which is why the two figures differ in
    it.

    `tasks` and `grouping` are passed to every panel (see
    :func:`plot_cost_accuracy`), and `title` goes over the whole figure: the
    per-data-set figures of section 3.3 use all three.

    `y_nbins` caps the y tick ladder of every panel. The default suits the
    appendix line; a flatter main figure passes fewer, or `-10.0` to `0.0` in
    steps of 2.5 runs together on a panel 0.4 in tall.

    `col_gap`, in inches, packs the columns closer than constrained layout
    does: every gap shrinks to what the labels on either side of it need, plus
    `col_gap`, and the panels take the width (see :func:`_compact_columns`).
    `None` leaves the layout's own spacing.
    """
    if legend not in ("slot", "row"):
        raise ValueError(f"legend must be 'slot' or 'row', got {legend!r}")
    refs = {METRICS[m]["reference"] for m in metrics}
    if len(refs) > 1:
        # One denominator, or the cost axis stops meaning one thing: `x` is a
        # ratio to the *same* reference the score is normalized against, and a
        # grid built on two of them would put two different constructions on
        # one pair of axes. Mixed *scales* are fine -- see `IMPROVEMENT`.
        raise ValueError("shared axes need one reference on every panel; got "
                         f"{sorted(refs)}")
    reference = refs.pop()
    n_rows, n_cols = shape
    n_slots = n_rows * n_cols
    most = n_slots - 1 if legend == "slot" else n_slots
    if not 0 < len(metrics) <= most:
        raise ValueError(
            f"{n_rows}x{n_cols} with legend={legend!r} holds 1 to {most} "
            f"metrics; got {len(metrics)}"
            + (" -- a full grid has no slot for the key, so pass "
               "legend='row'" if legend == "slot" and len(metrics) == n_slots
               else ""))
    # Two units on the grid (per cent and nats/dim) means `sharey` cannot be
    # left to `plt.subplots`; see the docstring.
    mixed_units = len({METRICS[m]["scale"] for m in metrics}) > 1
    fig, axes = pc.subplots(n_rows, n_cols, aspect=aspect, columns=columns,
                            sharex=True,
                            sharey=False if mixed_units else sharey)
    if title:
        fig.suptitle(title)
    flat = list(np.ravel(axes))
    frames, stage1_frames, arms = {}, {}, {}
    for ax, metric in zip(flat, metrics):
        ax.set_title(GRID_TITLE.get(metric, METRICS[metric]["label"]))
        try:
            _, _, frame = plot_cost_accuracy(
                metric, strict=strict, ax=ax, legend=False, annotate=False,
                verbose=verbose, mode=mode, per_seed=per_seed,
                point_alpha=point_alpha, point_size=point_size,
                grouping=grouping, tasks=tasks,
                y_transform=IMPROVEMENT[METRICS[metric]["scale"]])
        except ValueError as exc:
            # A metric no cell can draw keeps its panel and says so. Dropping it
            # would silently renumber the grid and put a different metric where
            # the caption says this one is.
            # Two lines, and no y scale: on a panel a sixth of a text width
            # wide the one-line note ran over both borders, and the 0-1 ladder
            # matplotlib leaves on an empty axis reads as data.
            ax.text(0.5, 0.5, "no loss-\nmatched cell", transform=ax.transAxes,
                    ha="center", va="center", color="0.45", style="italic",
                    fontsize=CFG.annotation_size)
            ax.tick_params(axis="y", left=False, labelleft=False)
            ax.grid(False)
            print(f"[{metric}] empty: {exc}")
            continue
        frames[metric] = frame
        # The axis names are written once, below; the per-panel call sets its own.
        ax.set_xlabel("")
        ax.set_ylabel("")
        for fam, rank in zip(frame["family"], frame["rank"]):
            arms[(fam, rank)] = None
        if include_stage1:
            # Recosted per *task* -- a stage-1 timing belongs to one task -- and
            # grouped afterwards like every other arm, then put on the panel's
            # unit (`frame` above is already grouped and transformed).
            per_task, _ = cost_accuracy(
                metric, strict=strict, per_seed=per_seed,
                costs=(None if tasks is None
                       else COSTS[COSTS["task"].map(lambda t: t in tasks)]))
            sub = group_tasks(stage1_only_frame(per_task), grouping)
            if not sub.empty:
                sub = sub.assign(y=IMPROVEMENT[METRICS[metric]["scale"]](
                    sub["y"].to_numpy(dtype=float)))
                draw_arm(ax, sub, arm_color("ours_stage1", 0), mode=mode,
                         mean_min_n=1 if mode == "mean" else 2,
                         alpha=point_alpha, size=point_size)
                arms[("ours_stage1", 0)] = None
                stage1_frames[metric] = sub
    if not frames:
        raise ValueError("no metric in the grid has a single drawable cell")

    # The slots the metrics did not fill. The first of them takes the legend
    # when it is drawn there, and the rest are blanked: an empty pair of axes
    # with ticks and a frame reads as a panel whose data failed to draw.
    spare = range(len(metrics), n_slots)
    for i in spare:
        flat[i].set_axis_off()
    # `sharex` hides the tick labels of every panel that is not in the bottom
    # row, on the assumption that the panel below carries them. Under a spare
    # slot there is no such panel, so the column above it would lose its x
    # axis entirely: put the labels back.
    for i in spare:
        if 0 <= i - n_cols < len(metrics):
            flat[i - n_cols].tick_params(labelbottom=True)

    # One tick ladder over all the panels (and the stage-1 markers, when
    # present -- they can sit further right than any regular arm). Each call
    # above sized the ticks to its own frame and the last one would otherwise
    # win, cropping the ladder to whichever metric happened to be drawn last.
    x = pd.concat([drawn_x(f, mode) for f in frames.values()]
                  + [drawn_x(f, mode) for f in stage1_frames.values()])
    ticks = cost_ticks(x)
    flat[0].set_xticks(ticks)
    flat[0].set_xticklabels([f"{t:g}" + r"$\times$" for t in ticks])
    flat[0].set_xticks([], minor=True)
    # And a denser y ladder than the default picks at this height: left alone
    # it labels 0.9 and 1.0 only, which reads as if the panels span two
    # values. `steps` keeps the ticks on round numbers. Set on every drawn
    # panel rather than on the row-leading ones: panels that share a y-axis
    # all get the same ladder out of it, and panels that do not each need
    # their own.
    for ax in flat[:len(metrics)]:
        ax.yaxis.set_major_locator(
            plt.MaxNLocator(nbins=y_nbins, steps=[1, 2, 2.5, 5, 10]))
    # Smaller tick labels than the paper's default, for these figures only:
    # panels this narrow leave about 0.85 in each, and at the default size
    # `2x` and `3x` touch on a ladder that has to reach 3.2. Applied after the
    # ladders above, which create the labels.
    for ax in flat:
        ax.tick_params(labelsize=CFG.tick_size * 0.82)
        # A multiplier (ETTm2's NLL reaches -3.3e6 in section 3.3) is drawn
        # left-aligned over the panel's corner, where on a panel this narrow it
        # runs into the title; right-aligned it hangs over the tick labels.
        ax.yaxis.get_offset_text().set_horizontalalignment("right")

    # The same-unit panels of a row, linked now that every one of them has been
    # drawn into and has the limits its own data asked for.
    if mixed_units and sharey == "row":
        for row in range(n_rows):
            in_row = [(metrics[i], flat[i])
                      for i in range(row * n_cols, min((row + 1) * n_cols,
                                                       len(metrics)))
                      if metrics[i] in frames]
            for scale in sorted({METRICS[m]["scale"] for m, _ in in_row}):
                _link_y([ax for m, ax in in_row
                         if METRICS[m]["scale"] == scale])

    # The axis name. One `supylabel` on the margin while every panel carries
    # the same unit; with the log score on the grid there are two, so the
    # majority unit stays on the margin and the odd panel names its own axis.
    # A single margin label spanning two units would be wrong on one of the
    # panels, which is worse than a label inside the grid.
    drawn_scales = [METRICS[m]["scale"] for m in metrics if m in frames]
    major = max(set(drawn_scales), key=drawn_scales.count)
    sup = fig.supylabel(IMPROVEMENT_LABEL[major], fontsize=CFG.label_size)
    for i, (ax, metric) in enumerate(zip(flat, metrics)):
        if metric not in frames or METRICS[metric]["scale"] == major:
            continue
        # **On the right, when the panel is in the last column** -- which is
        # where the log score is put deliberately. A second scale inside the
        # grid costs its column the width of a label and a tick strip, so that
        # column comes out narrower than the one above it and the panels stop
        # lining up; hung off the figure's right margin it mirrors `supylabel`,
        # takes space no panel wanted, and reads as the secondary axis it is.
        # A panel that is not in the last column has no margin to hang from and
        # keeps the label on its left, misalignment and all.
        # No axis *name* -- see `IMPROVEMENT_LABEL`; the unit goes in the
        # caption. Only the numbers, and which side they go on follows from
        # whether this panel's row-mates carry numbers of their own:
        # * **`sharey` on**: they do not -- one tick strip serves the row. A
        #   second strip *inside* such a row costs its column the width of the
        #   labels, so the panels stop lining up with the row above; hung off the
        #   figure's right margin instead it takes space no panel wanted and reads
        #   as the secondary axis it is. Needs the panel to be in the last column
        #   to have a margin to hang from.
        # * **`sharey=False`** (both figures): every panel already
        #   carries its own numbers on its own left, so this one is not special
        #   and belongs with them. Nothing is gained by singling it out, and a
        #   lone right-hand axis in a row of left-hand ones reads as a second
        #   scale on the panel beside it.
        right = bool(sharey) and i % n_cols == n_cols - 1
        if right:
            ax.yaxis.tick_right()
        # `sharex`/`_link_y` may have taken these away; the panel that carries
        # its own unit has to carry its own numbers.
        ax.tick_params(labelleft=not right, labelright=right)
    proxies, labels = _legend_proxies(arms, reference, mode)
    legend_kw = dict(frameon=True, fancybox=True, framealpha=0.95,
                     edgecolor="0.8", fontsize=CFG.legend_size * 0.72,
                     handletextpad=0.25, borderpad=0.4, handlelength=1.0)
    # One line of text, in inches, so a strip below the panels keeps its height
    # as the requested panel `aspect` changes.
    LABEL_IN = 0.24

    if legend == "slot":
        footer_frac = _carve_footer(fig, LABEL_IN)
        if col_gap is not None:
            _compact_columns(fig, np.reshape(axes, shape), col_gap)
        center_x = _axes_center_x(fig, flat)
        # One column, in the free slot of the grid: the keys read top to bottom
        # in the same pairs `GRID_LEGEND_ORDER` groups them into, and the box
        # sits where a sixth panel would have been rather than in a strip that
        # costs the figure height.
        flat[len(metrics)].legend(proxies, labels, loc="center", ncol=1,
                                  labelspacing=0.4, **legend_kw)
    else:
        # A row of panels leaves no slot, so the key goes in a strip of its own
        # below the axis label -- one row, in `GRID_LEGEND_ORDER`, so it reads
        # left to right in the order the panels introduce the colours. Its
        # strip is carved out by hand for the same reason the label's is:
        # `fig.supxlabel` is placed at the very bottom of the canvas, *under*
        # an `outside lower center` legend, which puts the label and the key in
        # the wrong order and, at this figure height, on top of each other.
        ncol = legend_ncol or len(proxies)
        # The strip is sized to the rows the key will actually take, not to the
        # one row it took when there were five keys and the full text width:
        # six keys at `ncol=3` is two rows, and a strip built for one puts the
        # second on top of the axis label.
        n_key_rows = int(np.ceil(len(proxies) / ncol))
        LEGEND_IN = 0.20 * n_key_rows + 0.10
        footer_frac = _carve_footer(fig, LABEL_IN + LEGEND_IN)
        if col_gap is not None:
            _compact_columns(fig, np.reshape(axes, shape), col_gap)
        center_x = _axes_center_x(fig, flat)
        fig.legend(proxies, labels, loc="upper center",
                   bbox_to_anchor=(center_x, LEGEND_IN / fig.get_size_inches()[1]),
                   bbox_transform=fig.transFigure, ncol=ncol,
                   columnspacing=1.2, labelspacing=0.2, **legend_kw)

    # The label hangs from the bottom of the panels rather than being built up
    # from the canvas floor, so it sits right under the tick labels it belongs
    # to instead of floating with whatever slack the strip leaves beneath.
    fig.text(center_x, footer_frac - 0.012, "Computational cost",
             ha="center", va="top", fontsize=CFG.label_size)

    # Centre the margin label on the **panels**, which is not where
    # `fig.supylabel` leaves it: its default is the middle of the canvas, and
    # `_carve_footer` has just grown the canvas downwards for the axis label
    # and the key, so the default sits below the panels by however tall that
    # strip is (0.13 of the figure on the appendix layout -- the label hung off
    # the bottom-left panel instead of naming the row). `fig.supylabel` above
    # already reserved the margin it needs; this only moves it, once the axes
    # are done moving around, and then freezes the layout engine so a later
    # draw (`pc.save`) cannot recentre it back. With an engine *instance* that
    # does nothing, not `"none"`: under `pc.apply_style`, `savefig` restores
    # `"none"` to constrained layout, and the notebook's own display after the
    # save would lay the panels out again, over the footer and the packing.
    fig.canvas.draw()
    if n_rows > 1:
        # With two rows the label cannot sit centred on either one's scale --
        # they do not share a y-axis -- so it goes at the midpoint of the gap
        # between them, the one point on the shared left margin that belongs to
        # both equally.
        row0_bottom = min(ax.get_position().y0 for ax in flat[:n_cols])
        row1_top = max(ax.get_position().y1 for ax in flat[n_cols:2 * n_cols])
        sup.set_y((row0_bottom + row1_top) / 2)
    else:
        drawn = flat[:len(metrics)]
        sup.set_y((min(ax.get_position().y0 for ax in drawn)
                   + max(ax.get_position().y1 for ax in drawn)) / 2)
    sup.set_verticalalignment("center")
    fig.set_layout_engine(PlaceHolderLayoutEngine(True, True))

    if save_as:
        pc.save(fig, save_as)
    return fig, axes, frames

In [ ]:
# Figure 4 of the paper. One cross per arm and nothing else: the clouds the
# appendix-style figure shows are what the crosses summarize. Six metrics fill the
# 2x3, so the key goes in a strip below in two rows of three -- six across a
# single column would not fit on one. Every panel has its own y-axis
# (`sharey=False`), so the NLL panel's numbers are on its left like the rest,
# and the columns are packed to 0.06 in between a frame and the next panel's
# labels (`col_gap`): constrained layout alone left 0.46 in between frames for
# panels 0.57 in wide.
fig, axes, GRID_FRAMES = plot_cost_accuracy_grid(
    include_stage1=False, mode="mean", legend="row", legend_ncol=3,
    sharey=False, col_gap=0.06, save_as="runtime_cost_accuracy_grid.pdf",
    aspect=0.45, y_nbins=4)

### 3.2 The appendix figure: one dot per data set

The same six metrics and the same cells, with each cross replaced by the
markers it summarizes: **one dot per data set** (`TASK_GROUPING`; the unsaved
per-seed variant below it draws one per data set and seed), so what the main
figure compresses
into a median is on the page. The layout follows from that — six panels in a
line at the full text width, and the key in a single row beneath them, because
a row of six leaves no slot to put it in.

Every panel takes **its own scale**, as in the main figure; a single range over
all six would put MSE, which moves by a few per cent, on the same axis as the
variogram score, which runs from $-15$ to $+17$ once the per-data-set values are
shown rather than their median. The heights are therefore *not* comparable
across panels, here or in the main figure: the tick labels carry the
magnitudes.

The log score — the `NLL` panel — is last and keeps its own unit here too,
nats per coordinate, numbers without a name; the caption carries the unit, as
in 3.1. Its axis is on the **left**, with every other panel's, as in the main
figure. SB is absent from the panel for the same reason as in 3.1: no
density, so no log score. ⚠️ It draws the four degenerate data sets -- depth,
ETTm2 (through its multivariate tasks), weather and electricity -- down to
$-1.5 \times 10^{4}$ nats (electricity's SCORE), so the seven others -- the six
healthy ones and traffic, half a nat below the reference -- sit on the top edge
(see 3.1 and the coverage note in section 3).

In [ ]:
# The appendix-style figure. The same cells, one dot per data set (`TASK_GROUPING`), six panels
# abreast of the full text width with every panel on its own y-axis and the key
# in a single row underneath. Saved only with `save = True`.
save = False
fig, axes, GRID_FRAMES_TASKS = plot_cost_accuracy_grid(
    metrics=APPENDIX_METRICS, include_stage1=False, mode="points",
    shape=APPENDIX_SHAPE, legend="row",
    columns="double", aspect=0.16, sharey=False,
    save_as="runtime_cost_accuracy_grid_appendix.pdf" if save else None)

In [ ]:
# Figure 2 once more, with one dot per (data set, seed) rather than per data set, and
# the dots made translucent so the stacks stay readable. Not saved.
fig, axes, GRID_FRAMES_SEEDS = plot_cost_accuracy_grid(
    metrics=APPENDIX_METRICS, include_stage1=False, mode="points",
    shape=APPENDIX_SHAPE, legend="row", columns="double", aspect=0.16,
    sharey=False, per_seed=True, point_alpha=0.35, point_size=14)

### 3.3 Per data set: one dot per task

The appendix figure once per data set, restricted to that data set's tasks and
with the grouping switched off (`grouping=None`): every dot is one **task**, at
the median over its seeds as everywhere in section 3, instead of the one
data-set median the appendix figure draws. An ETT data set therefore puts four
dots per arm on each panel (two modes × two horizons), illness, weather, electricity
and traffic two (their horizons), ERA5, depth and EUPPBench one. That is the cost–accuracy
trade-off *inside* each data set, before `GROUP_STATISTIC` collapses it: whether
an arm's modes and horizons agree, and which of them its data-set dot on the
appendix figure is standing in for.

Both axes keep their per-task denominator — the diagonal arm *of that task* —
so the dots of one arm spread along the cost axis too, and the diagonal's own
dots all sit on the crossing of the guides. Every panel has its own y-axis, as
in 3.2. Data sets that are timed but have no synced scores get no figure (the
cell names them). With `save = True` in the cell each is saved as
`runtime_cost_accuracy_<data set>.pdf`, the data set's folder name (`etth1`,
`national_illness`, `nyu_bts`, ...), with no title: the file name and the
caption say which data set it is, and the height goes to the panels. It is off:
the stacked figures below are the ones written. The dots are drawn at `alpha=0.8` so that coinciding tasks
show as a darker stack rather than one dot.

In [ ]:
# Figure 2 once per data set: the same six panels, restricted to the data set's
# tasks, with one dot per task (the median over its seeds) rather than one per
# data set -- four per arm on an ETT data set, one on ERA5. Untitled, with the
# dots slightly translucent so overlapping tasks stay visible; saved per data
# set under its folder name only with `save = True`.
save = False
DRAWN_TASKS = set()
for _metric in APPENDIX_METRICS:
    _frame, _ = cost_accuracy(_metric)
    if not _frame.empty:
        DRAWN_TASKS |= set(_frame["task"])
DATASETS = sorted({task_group(t, "dataset") for t in DRAWN_TASKS}, key=task_order)
_undrawn = sorted({task_group(t, "dataset") for t in COSTS["task"]} - set(DATASETS),
                  key=task_order)
if _undrawn:
    print("timed but nothing to draw, no figure: "
          + ", ".join(group_label(g, tex=False) for g in _undrawn))

GRID_FRAMES_BY_DATASET = {}
for ds in DATASETS:
    tasks = {t for t in DRAWN_TASKS if task_group(t, "dataset") == ds}
    fig, axes, GRID_FRAMES_BY_DATASET[ds] = plot_cost_accuracy_grid(
        metrics=APPENDIX_METRICS, include_stage1=False, mode="points",
        shape=APPENDIX_SHAPE, legend="row", columns="double", aspect=0.13,
        sharey=False, tasks=tasks, grouping=None, point_alpha=0.8,
        save_as=f"runtime_cost_accuracy_{ds[1]}.pdf" if save else None)
    plt.show()

**The same figures stacked (Figures 9 and 10 of the paper).** The per-data-set
figures above in two figures of one row per data set: the eight time-series data
sets, in the order of the cost tables (`task_order`: electricity after weather,
and traffic last), and then ERA5, depth and EUPPBench. The metric names the
column over the top row, the data set names the row on the right margin
(reading top to bottom), a single "Improvement" on the left margin names the
y-axes of every row, and one key under the whole figure serves every row. The
unit (per cent over the diagonal; nats/dim on the NLL column) is left to the
caption, as in 3.1 and 3.2.

The time-series rows share one cost axis -- every one of them falls between
about $1\times$ and $2.5\times$ -- so only the bottom row carries its ticks and the
rows pack tight. The other three do not share one: depth reaches $3.2\times$
while ERA5 and EUPPBench sit between $0.85\times$ and $1.25\times$, and a
common ladder would squeeze two of the three rows into a corner of their
panels, so each row keeps its own; its `row_height` is lowered to match, so
that the panels come out as tall as the time-series ones (0.46 in) despite the
extra tick strips. Saved as `runtime_cost_accuracy_stacked_ts.pdf` (5.4 in tall,
against about 11.3 in for the eight separate figures) and
`runtime_cost_accuracy_stacked_other.pdf` (2.6 in, against 4.3 in).

In [ ]:
from matplotlib.layout_engine import PlaceHolderLayoutEngine


def dataset_tasks(ds):
    """Every timed task of data set `ds` (a `task_group(..., "dataset")`)."""
    return {t for t in COSTS["task"] if task_group(t, "dataset") == ds}


#: Row labels of the stacked figures, where they differ from `group_label`:
#: a rotated label has one row's height to fit in, which "NYU depth (BTS)"
#: and "ERA5 $t_{2\mathrm{m}}$" do not.
STACK_LABEL = {"era5": "ERA5", "nyu_bts": "Depth"}


def plot_cost_accuracy_stack(datasets, metrics=APPENDIX_METRICS, strict=True,
                             row_height=0.58, point_alpha=0.8, point_size=22,
                             sharex=True, save_as=None, verbose=False):
    """The per-data-set figures of section 3.3 stacked into one: a row each.

    Every row is the appendix line of six panels restricted to one data set's
    tasks, one dot per task (`grouping=None`), exactly as the separate figures
    draw it. What stacking changes is only what the rows can share: the metric
    names the column once, over the top row; the data set names the row from
    the right margin, reading top to bottom; one "Improvement" on the left
    margin names the y-axes of every row; and one key underneath serves every
    row. The unit is left to the caption, as the log score's already is.

    `sharex=True` puts every row on one cost ladder, so only the bottom row
    carries x tick labels and the rows pack tight -- at the price that a data
    set whose arms span a narrow cost band sits in part of the panel width.
    `sharex=False` gives each row its own ladder and a tick strip of its own.

    `row_height` is inches of figure per row, before the title strip on top
    and the label-and-key strip underneath.
    """
    refs = {METRICS[m]["reference"] for m in metrics}
    if len(refs) > 1:
        raise ValueError("shared axes need one reference on every panel; got "
                         f"{sorted(refs)}")
    reference = refs.pop()
    n_rows, n_cols = len(datasets), len(metrics)
    width = pc.width_of("double")
    # The top strip holds the column titles; with a shared ladder the bottom
    # row's tick labels as well, and without one every row pays for its own.
    height = n_rows * row_height + (0.35 if sharex else 0.2 + 0.15 * n_rows)
    fig, axes = pc.subplots(n_rows, n_cols, figsize=(width, height),
                            sharex="all" if sharex else "row", sharey=False,
                            squeeze=False)
    frames, arms = {}, {}
    for r, ds in enumerate(datasets):
        tasks = dataset_tasks(ds)
        for c, metric in enumerate(metrics):
            ax = axes[r, c]
            if r == 0:
                ax.set_title(GRID_TITLE.get(metric, METRICS[metric]["label"]))
            try:
                _, _, frame = plot_cost_accuracy(
                    metric, strict=strict, ax=ax, legend=False, annotate=False,
                    verbose=verbose, mode="points", point_alpha=point_alpha,
                    point_size=point_size, grouping=None, tasks=tasks,
                    y_transform=IMPROVEMENT[METRICS[metric]["scale"]])
            except ValueError:
                # The same note, and the same missing y scale, as the separate
                # figures -- on three lines, since a stacked panel is narrower
                # and "matched cell" ran over its right border. An empty panel
                # keeps its slot so the column still means its metric.
                ax.text(0.5, 0.5, "no loss-\nmatched\ncell", transform=ax.transAxes,
                        ha="center", va="center", color="0.45", style="italic",
                        fontsize=CFG.annotation_size)
                ax.tick_params(axis="y", left=False, labelleft=False)
                ax.grid(False)
                continue
            frames[(ds, metric)] = frame
            ax.set_xlabel("")
            ax.set_ylabel("")
            for fam, rank in zip(frame["family"], frame["rank"]):
                arms[(fam, rank)] = None
        # The row's name on the right, turned the other way from the axis
        # name on the left, so the two margins cannot be read as one label.
        axes[r, -1].yaxis.set_label_position("right")
        axes[r, -1].set_ylabel(STACK_LABEL.get(ds[1], group_label(ds)),
                               fontsize=CFG.label_size, rotation=270,
                               va="bottom")
    if not frames:
        raise ValueError("no data set in the stack has a single drawable cell")

    # One axis name for every row, on the left margin, in the majority unit as
    # in `plot_cost_accuracy_grid`; the log score's column goes unnamed there
    # too. Created before the layout runs, so that it reserves its margin.
    scales = [METRICS[m]["scale"] for m in metrics]
    sup = fig.supylabel(IMPROVEMENT_LABEL[max(set(scales), key=scales.count)],
                        fontsize=CFG.label_size)

    # One ladder per shared x-axis: the whole figure, or each row.
    ladders = ([list(datasets)] if sharex else [[ds] for ds in datasets])
    for group in ladders:
        x = pd.concat([drawn_x(f, "points") for (ds, _), f in frames.items()
                       if ds in group])
        ticks = cost_ticks(x)
        ax = axes[datasets.index(group[0]), 0]
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{t:g}" + r"$\times$" for t in ticks])
        ax.set_xticks([], minor=True)
        if not sharex:
            for a in axes[datasets.index(group[0])]:
                a.tick_params(labelbottom=True)
    for ax in axes.ravel():
        ax.yaxis.set_major_locator(
            plt.MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]))
        ax.tick_params(labelsize=CFG.tick_size * 0.82)

    # A y multiplier kept out of the layout. ETTm2's NLL reaches -3.3e6, and
    # the offset text over that one panel otherwise opens a gap between two
    # whole rows. `set_in_layout(False)` does not reach it -- the axis counts
    # its offset text into its own bounding box whenever it is visible -- so
    # the real one is hidden and the same string is drawn in its place, as a
    # plain text that the layout can be told to skip. It hangs over the
    # panel's top-left corner, in the strip between the rows.
    fig.canvas.draw()
    for ax in axes.ravel():
        offset = ax.yaxis.get_major_formatter().get_offset()
        if not offset:
            continue
        ax.yaxis.get_offset_text().set_visible(False)
        ax.text(0.0, 1.0, offset, transform=ax.transAxes, ha="right",
                va="bottom", fontsize=CFG.tick_size * 0.82).set_in_layout(False)

    # The label and the key in a strip of their own, as in the separate
    # figures (see `plot_cost_accuracy_grid`), one key row for all the rows.
    proxies, labels = _legend_proxies(arms, reference, "points")
    # Tighter than the separate figures' 0.24 + 0.30, which leave 0.15 in
    # between the label and the key and 0.11 in under it; here the key sits
    # 0.06 in under the label and 0.03 in above the edge. (`LEGEND_IN` is where
    # the key's *anchor* goes; its border pad puts the box 0.04 in lower.)
    LABEL_IN, LEGEND_IN = 0.15, 0.22
    footer_frac = _carve_footer(fig, LABEL_IN + LEGEND_IN)
    center_x = _axes_center_x(fig, axes.ravel())
    fig.legend(proxies, labels, loc="upper center",
               bbox_to_anchor=(center_x, LEGEND_IN / fig.get_size_inches()[1]),
               bbox_transform=fig.transFigure, ncol=len(proxies),
               columnspacing=1.2, labelspacing=0.2, frameon=True, fancybox=True,
               framealpha=0.95, edgecolor="0.8",
               fontsize=CFG.legend_size * 0.72, handletextpad=0.25,
               borderpad=0.4, handlelength=1.0)
    fig.text(center_x, footer_frac - 0.02 / fig.get_size_inches()[1],
             "Computational cost", ha="center", va="top",
             fontsize=CFG.label_size)
    fig.canvas.draw()
    # Centred on the panels, not on the canvas that `_carve_footer` grew
    # downwards for the strip (see `plot_cost_accuracy_grid`).
    sup.set_y((min(ax.get_position().y0 for ax in axes.ravel())
               + max(ax.get_position().y1 for ax in axes.ravel())) / 2)
    sup.set_verticalalignment("center")
    # An engine *instance* that does nothing: `"none"` is restored to
    # constrained layout after `savefig` under `pc.apply_style`, which would
    # re-lay the panels over the strip on the next draw.
    fig.set_layout_engine(PlaceHolderLayoutEngine(True, True))
    if save_as:
        pc.save(fig, save_as)
    return fig, axes, frames

In [ ]:
# The per-data-set figures stacked, one row per data set (Figures 9 and 10): the
# time-series data sets on one shared cost axis, then ERA5, depth and EUPPBench
# with a cost axis per row (their cost ranges differ by a factor of three).
STACK_TS = [ds for ds in DATASETS if ds[0] == "time_series"]
STACK_OTHER = [ds for name in ("era5", "nyu_bts", "euppbench")
               for ds in DATASETS if ds[1] == name]
fig, axes, STACK_FRAMES_TS = plot_cost_accuracy_stack(
    STACK_TS, save_as="runtime_cost_accuracy_stacked_ts.pdf")
plt.show()
# `row_height` is smaller than the default because every row here pays for its
# own tick strip, which the layout otherwise hands to the panels: 0.535 makes
# them 0.46 in tall, the height of the time-series rows above (0.51 at 0.58).
fig, axes, STACK_FRAMES_OTHER = plot_cost_accuracy_stack(
    STACK_OTHER, sharex=False, row_height=0.535,
    save_as="runtime_cost_accuracy_stacked_other.pdf")
plt.show()

### 3.4 The objective-mismatched cells

The figures above are loss-matched: a point exists only where the synced run was
fitted with the objective its timing was measured under. That is strict enough to
drop several cells — most visibly the ERA5 low-rank arm at $r=16$, which exists
only as an NLL fit while the benchmark timed the kernel-score one.

Below are the cells that admitting `strict=False` would add, as a table rather
than as a third figure. They are the more complete picture of the runs and
the *less* comparable one: on ERA5 the NLL-fitted low-rank core scores a CRPS of
0.96 against the KS-fitted one's 0.15, so the extra spread is a fact about the
objective, not about the method or its cost. `plot_cost_accuracy(..., strict=False)`
draws them hollow if they are ever wanted on a panel.

In [ ]:
# The dropped cells themselves, rather than a figure of them: the notebook
# writes the figures above and nothing else.
LOOSE, _ = cost_accuracy("crps", strict=False, verbose=False)
LOOSE[~LOOSE["objective_matched"]][
    ["task_label", "arm_label", "slug", "objective", "raw", "x", "y"]]

## 4. Self-check

The transformations between the files on disk and the figure are all arithmetic,
so they can be checked against a synthetic campaign whose answer is known: an arm
built to be exactly twice as slow and to score exactly half must land on
$(2\times, 0.5)$ from every task, whatever that task's units and step cost.
The cross is checked against the artist rather than against the function that
computes it, so what is verified is where the mean was actually drawn — and,
for the same reason, the two figures are checked by looking at what they put on
the page: one glyph each, keyed in the glyph it draws.

In [ ]:
def _check_pm():
    """`pm` must print the mean and the spread to `DIGITS` decimal places."""
    cases = [(1.5231, 0.0034, r"$1.523 \pm 0.003$"),
             (0.010652, 0.000125, r"$0.011 \pm 0.000$"),
             (0.271653, 0.000911, r"$0.272 \pm 0.001$"),
             (93.139, 1.046, r"$93.139 \pm 1.046$")]
    for mean, sd, want in cases:
        got = pm(mean, sd)
        assert got == want, f"pm({mean}, {sd}) = {got!r}, want {want!r}"
    assert pm(np.nan, 1.0) is None
    assert pm(1.2341, np.nan) == r"$1.234$", pm(1.2341, np.nan)
    # The ms/step table: whole-millisecond mean, deviation at DIGITS, bold.
    assert pm(186.833, 0.301, mean_digits=0) == r"$187 \pm 0.301$"
    assert pm(186.833, 0.301, mean_digits=0, bold=True) == \
        r"$\mathbf{187 \pm 0.301}$"
    assert pm(186.833, 0.301, mean_digits=0, italic=True) == \
        r"$\mathit{187 \pm 0.301}$"
    assert pm(186.833, 0.301, mean_digits=0, bold=True, italic=True) == \
        r"{\boldmath$\mathit{187 \pm 0.301}$}"
    return f"pm formats mean and sd to {DIGITS} decimal places"


def _check_center_marker():
    """The cross has to sit on `TASK_STATISTIC` of the markers it summarises.

    Checked against the artist rather than against `arm_centers`, because what
    a reader takes off the panel is where the cross was *drawn*. The three
    tasks below are built so that mean and median disagree on both axes -- a
    case with an even count and no outlier would pass for either, so it would
    not notice the constant being read in one place and ignored in another.
    The expected position is computed from `TASK_STATISTIC` rather than written
    in, so this keeps testing the wiring when the constant is changed (it has
    been both ways: mean until 2026-09-08, median until 2026-09-09, mean since).
    """
    sub = pd.DataFrame(dict(x=[1.0, 2.0, 9.0], y=[0.90, 0.80, 0.10],
                            objective_matched=[True, True, True]))
    assert not np.isclose(np.mean(sub["x"]), np.median(sub["x"])) and \
        not np.isclose(np.mean(sub["y"]), np.median(sub["y"])), \
        "the case has to separate the median from the mean to test for it"
    fig, ax = plt.subplots()
    drawn = draw_arm(ax, sub, "k")
    crosses = [ln for ln in ax.lines if ln.get_marker() == CENTER_MARKER]
    assert drawn and len(crosses) == 1, "one cross per arm with >= 2 tasks"
    (cx,), (cy,) = crosses[0].get_xdata(), crosses[0].get_ydata()
    want_x, want_y = task_center(sub["x"]), task_center(sub["y"])
    assert np.isclose(cx, want_x) and np.isclose(cy, want_y), \
        (TASK_STATISTIC, (cx, cy), (want_x, want_y))
    # And the summary the panel draws is the one `arm_centers` reports, which
    # is what the notebook prints under the figure: two answers to the same
    # question is the failure a check against the artist alone cannot see.
    centers = arm_centers(sub.assign(family="ours", rank=0, task=[1, 2, 3]))
    assert np.isclose(centers["x"].iloc[0], cx) and \
        np.isclose(centers["y"].iloc[0], cy), "the table and the glyph disagree"
    # Every marker is the same shape, whichever arm it belongs to.
    assert {c.get_paths()[0].vertices.shape
            for c in ax.collections} == {ax.collections[0].get_paths()[0].vertices.shape}
    plt.close(fig)

    # A single-task arm gets its point and no cross: any statistic of one
    # number is that number, and a second glyph on top of it claims a summary.
    fig, ax = plt.subplots()
    drawn = draw_arm(ax, sub.iloc[:1], "k")
    assert not drawn and not [ln for ln in ax.lines
                              if ln.get_marker() == CENTER_MARKER]
    plt.close(fig)

    # And each one-glyph mode draws that glyph and nothing else.
    for mode, n_lines, n_collections in [("mean", 1, 0), ("points", 0, 1)]:
        fig, ax = plt.subplots()
        drawn = draw_arm(ax, sub, "k", mode=mode)
        assert drawn == (mode == "mean"), mode
        assert len([ln for ln in ax.lines
                    if ln.get_marker() == CENTER_MARKER]) == n_lines, mode
        assert len(ax.collections) == n_collections, mode
        plt.close(fig)
    return (f"the cross is drawn at the {TASK_STATISTIC} of the markers and "
            "agrees with `arm_centers`, only where there are at least two, and "
            "each one-glyph mode draws that glyph alone")


def _check_axes():
    """A synthetic campaign with a known answer, in the schema the readers return.

    Two tasks with wildly different units and step costs; one arm built to be
    exactly twice as slow as the *diagonal* reference and to score exactly half
    of it. Both normalizations have to remove the unit and the scale.

    The deterministic arm is present in the synthetic costs on purpose, at a
    cost and a score that would be conspicuous on the panel: it is there to
    check that `PANEL_EXCLUDE` keeps it off.
    """
    tasks = [("synthetic", "a", None, None), ("synthetic", "b", None, None)]
    scale = {tasks[0]: 0.05, tasks[1]: 7.0}
    step = {tasks[0]: 11.0, tasks[1]: 640.0}
    costs, scores = [], []
    for task in tasks:
        # Multipliers are against the diagonal arm, which is built as exactly
        # 1.0x and 1.0 so the assertions read straight off the construction.
        for fam, rank, cost_mult, score_mult, obj in [
                ("deterministic", 0, 0.7, 1.4, "mse"),
                ("phys_diag", 0, 1.0, 1.0, "kernel_score"),
                ("ours", 0, 2.0, 0.5, "kernel_score")]:
            costs.append(dict(
                task=task, family=fam, rank=rank, n_seeds=3, status="ok",
                method=fam, transform=None, staged=False, batch_size=8,
                event_dim=100, peak_gb=1.0,
                s_per_step=step[task] * cost_mult / 1e3, s_per_step_sd=0.0,
                steps_per_s=1e3 / (step[task] * cost_mult), steps_per_s_sd=0.0,
                diag_s_per_step=step[task] / 1e3, rel_to_diag=cost_mult,
                overhead_pct=100 * (cost_mult - 1),
                task_label=task[1], arm_label=fam))
            for seed in (1, 2, 3):
                scores.append(dict(
                    task=task, slug=f"{fam}_{obj}", family=fam, method=fam,
                    rank=rank, stage1=None, objective=obj, seed=str(seed),
                    crps=scale[task] * score_mult, nll=np.nan,
                    energy_score=scale[task] * score_mult,
                    kernel_score=scale[task] * score_mult,
                    variogram_score=np.nan, mse=scale[task] * score_mult,
                    mae=np.nan, d=100, nll_per_d=np.nan))
    frame, _ = cost_accuracy("crps", strict=True,
                             costs=pd.DataFrame(costs),
                             scores=pd.DataFrame(scores))
    ours = frame[frame["family"] == "ours"]
    assert len(ours) == 2, f"both synthetic tasks should contribute, got {len(ours)}"
    assert np.allclose(ours["x"], 2.0), ours["x"].tolist()
    assert np.allclose(ours["overhead_pct"], 100.0), ours["overhead_pct"].tolist()
    assert np.allclose(ours["y"], 0.5), ours["y"].tolist()
    diag = frame[frame["family"] == "phys_diag"]
    assert np.allclose(diag["x"], 1.0) and np.allclose(diag["y"], 1.0), \
        "the reference arm must land on the crossing of the guides"
    assert frame[frame["family"].isin(PANEL_EXCLUDE)].empty, \
        f"excluded families reached the panel: {sorted(set(frame['family']) & PANEL_EXCLUDE)}"
    return ("cost and score normalizations remove the unit and the scale: "
            "an arm 2x slower at half the score lands on (2x, 0.5) from "
            "both synthetic tasks, whose units differ by 140x and whose step "
            "costs differ by 58x; the reference lands on (1x, 1) and the "
            "excluded families do not land at all")

def _check_cost_ticks():
    """The cost ladder has to cover the range it is drawn over.

    A fixed locator pulls the view out to its last tick, so a ladder that
    overshoots the data leaves the panel part empty and one that stops short of
    it cannot: the assertions below are the two ranges the paper's figures
    actually ask for, wide (the appendix's per-task points) and narrow (the
    main figure's means, which span less than an octave).
    """
    assert cost_ticks([0.85, 3.15]) == [1, 2, 3], cost_ticks([0.85, 3.15])
    assert cost_ticks([0.86, 1.81]) == [1, 1.5], cost_ticks([0.86, 1.81])
    assert cost_ticks([1.0, 1.0]) == [1], cost_ticks([1.0, 1.0])
    return ("the cost ladder is coarse where the arms span an octave and "
            "admits 1.5x only where it would otherwise label one tick")


def _legend_of(fig):
    """The figure's one legend, wherever it was attached."""
    found = list(fig.legends) + [ax.get_legend() for ax in fig.axes
                                 if ax.get_legend() is not None]
    assert len(found) == 1, f"expected one legend, found {len(found)}"
    return found[0]


def _check_figure_glyphs():
    """Each figure draws one glyph, and keys the one it draws.

    Checked on the artists of the real figures rather than on the arguments
    they were built with: what a reader takes off the page is what was drawn,
    and the failure this guards against -- a key of dots over a panel of
    crosses -- is invisible to any check that stops at the call.
    """
    main, main_axes, main_frames = plot_cost_accuracy_grid(
        include_stage1=False, mode="mean", legend="row", legend_ncol=3,
        sharey=False)
    panels = list(np.ravel(main_axes))[:len(GRID_METRICS)]
    assert not any(ax.collections for ax in panels), \
        "the main figure draws no per-task markers"
    assert all([ln for ln in ax.lines if ln.get_marker() == CENTER_MARKER]
               for ax in panels), "every panel of the main figure draws crosses"
    leg = _legend_of(main)
    assert {h.get_marker() for h in leg.legend_handles} == {CENTER_MARKER}, \
        "the main figure's key is drawn in crosses"
    n_arms = len(leg.legend_handles)
    plt.close(main)

    app, app_axes, _ = plot_cost_accuracy_grid(
        include_stage1=False, mode="points", shape=APPENDIX_SHAPE, legend="row",
        columns="double", aspect=0.20, sharey=False)
    panels = list(np.ravel(app_axes))
    assert all(ax.collections for ax in panels), \
        "every panel of the appendix figure draws its tasks"
    assert not any(ln.get_marker() == CENTER_MARKER
                   for ax in panels for ln in ax.lines), \
        "the appendix figure draws no means"
    leg = _legend_of(app)
    assert {h.get_marker() for h in leg.legend_handles} == {POINT_MARKER}, \
        "the appendix figure's key is drawn in dots"
    assert len(leg.legend_handles) == n_arms, \
        "both figures key the same arms"
    # One row, measured off the drawn keys rather than off the `ncol` it was
    # asked for: `ncol` is a maximum, and a legend too wide for its strip wraps
    # without complaining. The tolerance is half a line, because the boxes of
    # two labels on the same row still differ by a descender. Asserted for the
    # appendix figure only -- the main one deliberately takes two rows of three,
    # since six keys do not fit across a single column.
    app.canvas.draw()
    boxes = [t.get_window_extent() for t in leg.get_texts()]
    spread = max(b.y0 for b in boxes) - min(b.y0 for b in boxes)
    assert spread < 0.5 * np.median([b.height for b in boxes]), \
        f"the appendix key must be one row; its labels span {spread:.0f} px"
    plt.close(app)
    return (f"the main figure draws and keys {n_arms} crosses, the appendix "
            f"figure draws and keys {n_arms} dots in a single row")


for check in (_check_pm, _check_center_marker, _check_cost_ticks, _check_axes,
              _check_figure_glyphs):
    print("PASS:", check())